In [ ]:
from pathlib import Path

# Works when the notebook is launched from either the project root
# or the notebooks directory.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROJECT_ROOT


# NLP Coursework: Sequence Classification across Varieties of English
Notebook structure:
- Q1 Data analysis and visualisation
- Q2.1 Classical baselines vs fine-tuned Transformer
- Q2.2 Cross-variety evaluation matrix
- Q2.3 LoRA adapters on an open-weight LLM
- Q3 Evaluation metrics and confusion matrices
- Q4 Error analysis and few-shot prompting
- Q5 Deployment and efficiency


## Stage 1: Dataset analysis and visualisation

In [ ]:
import os
from datasets import load_dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import fisher_exact
import os
os.environ.setdefault("HF_HOME", os.path.expanduser("~/.cache/huggingface"))


In [ ]:
# display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
sns.set_style("whitegrid")

In [ ]:
# load dataset
ds = load_dataset("surrey-nlp/BESSTIE-CW-26")
print(ds)

In [ ]:
train = ds['train']
print("First 3 training examples:")
for i in range(3):
    print(f"\n[{i}] {train[i]}")

In [ ]:
print("unique values in key columns:")
print(f" variety: {sorted(set(train['variety']))}")
print(f" source: {sorted(set(train['source']))}")
print(f" Sentiment: {sorted(set(train['Sentiment']))}")
print(f" Sarcasm: {sorted(set(train['Sarcasm']))}")

In [ ]:
train_df = ds['train'].to_pandas()
val_df   = ds['validation'].to_pandas()
test_df  = ds['test'].to_pandas()

In [ ]:
# casting labels to ints
for df in (train_df, val_df, test_df):
    df['Sentiment'] = df['Sentiment'].astype(int)
    df['Sarcasm']   = df['Sarcasm'].astype(int)

In [ ]:
print(f"Train: {train_df.shape}")
print(f"Val:   {val_df.shape}")
print(f"Test:  {test_df.shape}")

In [ ]:
train_df.head()

In [ ]:
# tagging each split
train_df['split'] = 'train'
val_df['split']   = 'validation'
test_df['split']  = 'test'
all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
print(f"Total instances across all splits: {len(all_df)}\n")

In [ ]:
print("Counts by variety x split:")
variety_split = all_df.groupby(['variety', 'split']).size().unstack(fill_value=0)
variety_split = variety_split[['train', 'validation', 'test']]
variety_split['TOTAL'] = variety_split.sum(axis=1)
print(variety_split)

In [ ]:
# variety x source counts
print("Counts by variety x source (Google Places vs Reddit):")
variety_source = all_df.groupby(['variety', 'source']).size().unstack(fill_value=0)
variety_source['TOTAL'] = variety_source.sum(axis=1)
print(variety_source)

In [ ]:
variety_source_pct = variety_source.drop(columns='TOTAL').div(variety_source['TOTAL'] - variety_source.get('TOTAL', 0), axis=0) * 100

In [ ]:
# clean recompute
totals = variety_source.drop(columns='TOTAL').sum(axis=1)
variety_source_pct = variety_source.drop(columns='TOTAL').div(totals, axis=0) * 100
print(variety_source_pct.round(1))

In [ ]:
print("Sentiment distribution per variety (counts):")
sent_counts = all_df.groupby(['variety', 'Sentiment']).size().unstack(fill_value=0)
sent_counts.columns = ['Negative (0)', 'Positive (1)']
sent_counts['TOTAL'] = sent_counts.sum(axis=1)
sent_counts['% Positive'] = (sent_counts['Positive (1)'] / sent_counts['TOTAL'] * 100).round(1)
print(sent_counts)

In [ ]:
print("Sarcasm distribution per variety (counts):")
sarc_counts = all_df.groupby(['variety', 'Sarcasm']).size().unstack(fill_value=0)
sarc_counts.columns = ['Not sarcastic (0)', 'Sarcastic (1)']
sarc_counts['TOTAL'] = sarc_counts.sum(axis=1)
sarc_counts['% Sarcastic'] = (sarc_counts['Sarcastic (1)'] / sarc_counts['TOTAL'] * 100).round(1)
print(sarc_counts)

In [ ]:
# For training split only, this matters for class-weight decisions
sarc_train = train_df.groupby(['variety', 'Sarcasm']).size().unstack(fill_value=0)
sarc_train.columns = ['Not sarcastic (0)', 'Sarcastic (1)']
sarc_train['TOTAL'] = sarc_train.sum(axis=1)
sarc_train['% Sarcastic'] = (sarc_train['Sarcastic (1)'] / sarc_train['TOTAL'] * 100).round(1)
sarc_train['Pos:Neg ratio'] = (sarc_train['Not sarcastic (0)'] / sarc_train['Sarcastic (1)']).round(2)
print(sarc_train)

In [ ]:
print("Sarcasm rate by variety x source:")
sarc_by_source = all_df.groupby(['variety', 'source'])['Sarcasm'].agg(['count', 'sum', 'mean'])
sarc_by_source.columns = ['N', 'Sarcastic_count', 'Sarcasm_rate']
sarc_by_source['Sarcasm_rate'] = (sarc_by_source['Sarcasm_rate'] * 100).round(1)
print(sarc_by_source)

In [ ]:
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

In [ ]:
VARIETY_COLORS = {'en-UK': '#1f77b4','en-AU': '#2ca02c','en-IN': '#d62728',}
VARIETY_ORDER = ['en-UK', 'en-AU', 'en-IN']

In [ ]:
# plot style
plt.rcParams.update({'figure.dpi': 110,'savefig.dpi': 200,'savefig.bbox': 'tight','font.size': 10,'axes.titlesize': 11,'axes.labelsize': 10,'axes.spines.top': False,'axes.spines.right': False,})

In [ ]:
# save figure to figuresname.png
def save_fig(name):
    path = FIG_DIR / f"{name}.png"
    plt.savefig(path, bbox_inches='tight')
    print(f"Saved:{path}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sent = all_df.groupby(['variety', 'Sentiment']).size().unstack(fill_value=0)
sent = sent.loc[VARIETY_ORDER]
sent_pct = sent.div(sent.sum(axis=1), axis=0) * 100
ax = axes[0]
sent_pct[0].plot(kind='bar', ax=ax, color='#e0a458', label='Negative (0)', width=0.7)
sent_pct[1].plot(kind='bar', ax=ax, color='#5b8ca8',bottom=sent_pct[0], label='Positive (1)', width=0.7)
for i, variety in enumerate(VARIETY_ORDER):
    neg_n, pos_n = sent.loc[variety, 0], sent.loc[variety, 1]
ax.text(i, sent_pct.loc[variety, 0] / 2, f"n={neg_n}",ha='center', va='center', fontsize=9, color='white', fontweight='bold')ax.text(i, sent_pct.loc[variety, 0] + sent_pct.loc[variety, 1] / 2,
            f"n={pos_n}", ha='center', va='center', fontsize=9, color='white', fontweight='bold')
ax.set_title("Sentiment distribution by variety")
ax.set_ylabel("Proportion (%)")
ax.set_xlabel("")
ax.set_ylim(0, 100)
ax.set_xticklabels(VARIETY_ORDER, rotation=0)
ax.legend(loc='lower right', framealpha=0.9)
sarc = all_df.groupby(['variety', 'Sarcasm']).size().unstack(fill_value=0)
sarc = sarc.loc[VARIETY_ORDER]
sarc_pct = sarc.div(sarc.sum(axis=1), axis=0) * 100
ax = axes[1]
sarc_pct[0].plot(kind='bar', ax=ax, color='#9fb8ad', label='Not sarcastic (0)', width=0.7)
sarc_pct[1].plot(kind='bar', ax=ax, color='#c44536',bottom=sarc_pct[0], label='Sarcastic (1)', width=0.7)
for i, variety in enumerate(VARIETY_ORDER):
    neg_n, pos_n = sarc.loc[variety, 0], sarc.loc[variety, 1]
    ax.text(i, sarc_pct.loc[variety, 0] / 2, f"n={neg_n}",ha='center', va='center', fontsize=9, color='white', fontweight='bold')
    ax.text(i, sarc_pct.loc[variety, 0] + sarc_pct.loc[variety, 1] / 2,f"n={pos_n}", ha='center', va='center', fontsize=8, color='white', fontweight='bold')
ax.set_title("Sarcasm distribution by variety")
ax.set_ylabel("Proportion (%)")
ax.set_xlabel("")
ax.set_ylim(0, 100)
ax.set_xticklabels(VARIETY_ORDER, rotation=0)
ax.legend(loc='lower right', framealpha=0.9)
plt.suptitle("Label distributions across varieties (all splits combined)",fontsize=12, y=1.02)
save_fig("fig1_label_distributions")
plt.show()

In [ ]:
sarc_rates = all_df.groupby(['variety', 'source']).agg(N=('Sarcasm', 'size'),sarc_count=('Sarcasm', 'sum')).reset_index()
sarc_rates['sarc_rate_pct'] = (sarc_rates['sarc_count'] / sarc_rates['N'] * 100).round(2)
pivot_rates = sarc_rates.pivot(index='variety', columns='source', values='sarc_rate_pct')
pivot_counts = sarc_rates.pivot(index='variety', columns='source', values='sarc_count')
pivot_N = sarc_rates.pivot(index='variety', columns='source', values='N')
pivot_rates = pivot_rates.loc[VARIETY_ORDER]
pivot_counts = pivot_counts.loc[VARIETY_ORDER]
pivot_N = pivot_N.loc[VARIETY_ORDER]
fig, ax = plt.subplots(figsize=(8.5, 4.5))
x = np.arange(len(VARIETY_ORDER))
bar_w = 0.36
google_bars = ax.bar(x - bar_w/2, pivot_rates['Google'], bar_w,label='Google Places', color='#4c78a8', edgecolor='white')
reddit_bars = ax.bar(x + bar_w/2, pivot_rates['Reddit'], bar_w,label='Reddit', color='#e45756', edgecolor='white')
for i, variety in enumerate(VARIETY_ORDER):
    for source, bars, offset in [('Google', google_bars, -bar_w/2),('Reddit', reddit_bars,  bar_w/2)]:
        rate = pivot_rates.loc[variety, source]
        n_sarc = int(pivot_counts.loc[variety, source])
        n_total = int(pivot_N.loc[variety, source])
        ax.text(i + offset, rate + 0.8,f"{rate:.1f}%\n({n_sarc}/{n_total})",ha='center', va='bottom', fontsize=8.5)
ax.set_xticks(x)
ax.set_xticklabels(VARIETY_ORDER)
ax.set_ylabel("Sarcasm rate (%)")
ax.set_title("Sarcasm rate by variety and source domain\n"
             "Sarcasm is a Reddit phenomenon Google reviews contain almost none",fontsize=11)
ax.set_ylim(0, max(pivot_rates.values.flatten()) + 8)
ax.legend(loc='upper right', framealpha=0.95)
save_fig("fig2_sarcasm_by_source")
plt.show()

In [ ]:
all_df['word_count'] = all_df['text'].str.split().str.len()
print("Word count statistics per variety (all splits):")
length_stats = all_df.groupby('variety')['word_count'].agg(['count', 'mean', 'median', 'std', 'min', 'max',lambda x: x.quantile(0.95), lambda x: x.quantile(0.99)]).round(1)
length_stats.columns = ['N', 'mean', 'median', 'std', 'min', 'max', 'p95', 'p99']
print(length_stats)

In [ ]:
print("Word count statistics by variety x source:")
length_by_source = all_df.groupby(['variety', 'source'])['word_count'].agg(['count', 'mean', 'median',lambda x: x.quantile(0.95)]).round(1)
length_by_source.columns = ['N', 'mean', 'median', 'p95']
print(length_by_source)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
ax = axes[0]
bins = np.logspace(np.log10(3), np.log10(1400), 40)
for variety in VARIETY_ORDER:
    subset = all_df[all_df['variety'] == variety]['word_count']
    ax.hist(subset, bins=bins, alpha=0.55,label=f"{variety} (median={subset.median():.0f})",color=VARIETY_COLORS[variety], edgecolor='white', linewidth=0.5)
ax.set_xscale('log')
ax.set_xlabel("Word count (log scale)")
ax.set_ylabel("Number of examples")
ax.set_title("Text length distribution by variety")
ax.legend(loc='upper right', framealpha=0.95)
ax.axvline(256, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.text(256, ax.get_ylim()[1] * 0.95, "  256 tokens approx",fontsize=8, color='gray', va='top')
ax = axes[1]
y_cap = all_df['word_count'].quantile(0.99)
plot_df = all_df[all_df['word_count'] <= y_cap].copy()
plot_df['variety_source'] = plot_df['variety'] + "\n" + plot_df['source']
order = []
for v in VARIETY_ORDER:
    for s2 in ['Google', 'Reddit']:
        order.append(f"{v}\n{s2}")
sns.boxplot(data=plot_df, x='variety_source', y='word_count',order=order, ax=ax,palette=[VARIETY_COLORS[v] for v in VARIETY_ORDER for _ in range(2)],width=0.6, fliersize=2)
ax.set_xlabel("")
ax.set_ylabel("Word count")
ax.set_title("Length by variety x source (outliers above p99 clipped)")
ax.tick_params(axis='x', labelsize=9)
plt.tight_layout()
save_fig("fig3_text_length")
plt.show()

In [ ]:
# function to make lowercase, keeping words and apostrophes, splitting on whitespace
def simple_tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9'\s]", " ", text)
    tokens = text.split()
    tokens = [t for t in tokens if (len(t) > 1 or t in {'a', 'i'})]
    return tokens

In [ ]:
sample = train_df['text'].iloc[0]
print("Sample text:", sample[:200])
print("\nTokens:", simple_tokenize(sample)[:30])

In [ ]:
# build vocab per variety using only training data
variety_token_counts = {}
variety_vocabs_raw = {}

In [ ]:
for variety in VARIETY_ORDER:
    texts = train_df[train_df['variety'] == variety]['text'].tolist()
    all_tokens = []
    for t in texts:
        all_tokens.extend(simple_tokenize(t))
    counter = Counter(all_tokens)
    variety_token_counts[variety] = counter
    variety_vocabs_raw[variety] = set(counter.keys())

In [ ]:
print("Vocabulary sizes (training data only):")
for v in VARIETY_ORDER:
    total_tokens = sum(variety_token_counts[v].values())
    unique_tokens = len(variety_vocabs_raw[v])
    print(f"  {v}: {unique_tokens:>6} unique types, {total_tokens:>7} total tokens")

In [ ]:
def jaccard(a, b):
    return len(a & b) / len(a | b)

In [ ]:
# filter to words appearing at least 3 times
MIN_FREQ = 3
variety_vocabs_filtered = {v: {tok for tok, c in counts.items() if c >= MIN_FREQ}for v, counts in variety_token_counts.items()}

In [ ]:
print(f"Filtered vocabulary sizes (freq >= {MIN_FREQ}):")
for v in VARIETY_ORDER:
    print(f"  {v}: {len(variety_vocabs_filtered[v]):>5} types")

In [ ]:
print("Jaccard similarity (raw vocabulary):")
pairs = [('en-UK', 'en-AU'), ('en-UK', 'en-IN'), ('en-AU', 'en-IN')]
for a, b in pairs:
    j = jaccard(variety_vocabs_raw[a], variety_vocabs_raw[b])
    inter = len(variety_vocabs_raw[a] & variety_vocabs_raw[b])
    union = len(variety_vocabs_raw[a] | variety_vocabs_raw[b])
    print(f"{a}  {b}: Jaccard = {j:.3f} (|i|={inter}, |u|={union})")

In [ ]:
print(f"Jaccard similarity (frequency >= {MIN_FREQ}):")
for a, b in pairs:
    j = jaccard(variety_vocabs_filtered[a], variety_vocabs_filtered[b])
    inter = len(variety_vocabs_filtered[a] & variety_vocabs_filtered[b])
    union = len(variety_vocabs_filtered[a] | variety_vocabs_filtered[b])
    print(f"{a}  {b}: Jaccard = {j:.3f} (|i|={inter}, |u|={union})")

In [ ]:
# words appearing >=5 times in target but <=1 time in all others
def variety_exclusive_words(target, others, top_n=30):
    target_counts = variety_token_counts[target]
    exclusive = []
    for word, count in target_counts.items():
        if count < 5:
            continue
        if all(variety_token_counts[o].get(word, 0) <= 1 for o in others):
            exclusive.append((word, count))
    exclusive.sort(key=lambda x: -x[1])
    return exclusive[:top_n]

In [ ]:
print("Words distinctive to each variety (>=5 in target, <=1 in others):")
for v in VARIETY_ORDER:
    others = [o for o in VARIETY_ORDER if o != v]
    exclusive = variety_exclusive_words(v, others, top_n=25)
    print(f"\n{v} (vs {', '.join(others)}):")
    words_str = ', '.join([f"{w}({c})" for w, c in exclusive])
    print(f"  {words_str}")

In [ ]:
# concatenate all training texts per variety for TF-IDF
variety_docs = {}
for variety in VARIETY_ORDER:
    texts = train_df[train_df['variety'] == variety]['text'].tolist()
    variety_docs[variety] = " ".join(texts)

In [ ]:
docs = [variety_docs[v] for v in VARIETY_ORDER]

In [ ]:
vectorizer = TfidfVectorizer(lowercase=True,token_pattern=r"[a-z0-9']+",min_df=3,max_df=1.0,sublinear_tf=True,)
tfidf_matrix = vectorizer.fit_transform(docs)
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}   (3 varieties x vocab_size)")

In [ ]:
# cosine similarity matrix
cos_sim = cosine_similarity(tfidf_matrix)
cos_df = pd.DataFrame(cos_sim, index=VARIETY_ORDER, columns=VARIETY_ORDER)
print("TF-IDF cosine similarity matrix:")
print(cos_df.round(3))

In [ ]:
# pairwise numbers for comparison with Jaccard
print("Pairwise TF-IDF cosine similarity:")
for a, b in [('en-UK', 'en-AU'), ('en-UK', 'en-IN'), ('en-AU', 'en-IN')]:
    print(f"  {a} <-> {b}:  cosine = {cos_df.loc[a, b]:.3f}")

In [ ]:
jac_matrix = np.zeros((3, 3))
for i, a in enumerate(VARIETY_ORDER):
    for j, b in enumerate(VARIETY_ORDER):
        jac_matrix[i, j] = jaccard(variety_vocabs_filtered[a], variety_vocabs_filtered[b])
jac_df = pd.DataFrame(jac_matrix, index=VARIETY_ORDER, columns=VARIETY_ORDER)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.heatmap(jac_df, annot=True, fmt=".3f", cmap="Blues",vmin=0.35, vmax=1.0, cbar=True, ax=axes[0],linewidths=1, linecolor='white', square=True,annot_kws={"fontsize": 11})
axes[0].set_title("Jaccard similarity (filtered vocab, freq >= 3)")
sns.heatmap(cos_df, annot=True, fmt=".3f", cmap="Greens",vmin=0.90, vmax=1.0, cbar=True, ax=axes[1],linewidths=1, linecolor='white', square=True,annot_kws={"fontsize": 11})
axes[1].set_title("TF-IDF cosine similarity")
plt.suptitle("Inter-variety vocabulary similarity: two complementary views",fontsize=12, y=1.03)
plt.tight_layout()
save_fig("fig4_vocab_similarity_heatmaps")
plt.show()

In [ ]:
HINDI_FUNCTION_WORDS = {
    'hai', 'hain', 'tha', 'thi', 'the', 'hoga', 'hogi', 'hoge',
    'ho', 'hota', 'hoti', 'hote', 'raha', 'rahi', 'rahe',
    'ka', 'ki', 'ke', 'ko', 'se', 'me', 'mein', 'par', 'tak',
    'nahi', 'nahin', 'mat', 'na',
    'aur', 'ya', 'lekin', 'phir', 'toh', 'agar', 'kyun', 'kyunki',
    'bhi', 'hi', 'sab', 'sabhi',
    'yeh', 'ye', 'woh', 'wo', 'mera', 'meri', 'tera', 'teri',
    'apna', 'apni', 'hum', 'tum', 'unhe', 'inhe',
}

In [ ]:
INDIC_CONTENT_WORDS = {
    'dosa', 'idli', 'thali', 'biriyani', 'biryani', 'samosa',
    'paneer', 'masala', 'chai', 'momo', 'momos', 'dal', 'daal',
    'roti', 'paratha', 'pakora', 'jalebi', 'ladoo', 'mutton',
    'chole', 'kulfi', 'chutney', 'rasgulla', 'gulab', 'jamun',
    'tandoori', 'naan', 'kebab', 'tikka', 'korma',
    'lakh', 'lakhs', 'crore', 'crores', 'paisa', 'rupee', 'rupees', 'rs',
    'ji', 'bhaiya', 'didi', 'yaar', 'beta', 'aunty', 'uncle',
    'pandit', 'sahib', 'saab', 'madam', 'sir',
}

In [ ]:
print(f"Function-word lexicon size:  {len(HINDI_FUNCTION_WORDS)}")
print(f"Content-word lexicon size:   {len(INDIC_CONTENT_WORDS)}")

In [ ]:
# total count of lexicon items in a token counter
def count_lexicon(lexicon, counts_dict):
    return sum(counts_dict.get(w, 0) for w in lexicon)

In [ ]:
# per-variety totals for both lexicons
print("Lexicon counts per variety (raw occurrences in training data):")
print(f"{'Variety':<10}{'Function words':>18}{'Content words':>18}{'Tokens total':>18}")
for v in VARIETY_ORDER:
    func_n = count_lexicon(HINDI_FUNCTION_WORDS, variety_token_counts[v])
    cont_n = count_lexicon(INDIC_CONTENT_WORDS, variety_token_counts[v])
    total = sum(variety_token_counts[v].values())
    print(f"{v:<10}{func_n:>18}{cont_n:>18}{total:>18}")

In [ ]:
print("Function-word hits per variety (to check for false positives):")
for v in VARIETY_ORDER:
    hits = [(w, variety_token_counts[v].get(w, 0))
            for w in HINDI_FUNCTION_WORDS
            if variety_token_counts[v].get(w, 0) > 0]
    hits.sort(key=lambda x: -x[1])
    print(f"\n{v} (top 15):")
    for w, c in hits[:15]:
        print(f"  {w:<10} {c}")

In [ ]:
# these overlap with common English words -- remove them
HOMOGRAPH_CONTAMINATED = {'the', 'me', 'hi', 'na', 'par', 'ya', 'wo', 'se'}
HINDI_FUNCTION_WORDS_CLEAN = HINDI_FUNCTION_WORDS - HOMOGRAPH_CONTAMINATED

In [ ]:
print(f"Original function-word lexicon:  {len(HINDI_FUNCTION_WORDS)}")
print(f"Removed (homograph contamination): {sorted(HOMOGRAPH_CONTAMINATED)}")
print(f"Cleaned function-word lexicon:   {len(HINDI_FUNCTION_WORDS_CLEAN)}")

In [ ]:
print("CLEANED counts per variety:")
print(f"{'Variety':<10}{'Function words':>18}{'Content words':>18}{'Tokens total':>18}")
for v in VARIETY_ORDER:
    func_n = count_lexicon(HINDI_FUNCTION_WORDS_CLEAN, variety_token_counts[v])
    cont_n = count_lexicon(INDIC_CONTENT_WORDS, variety_token_counts[v])
    total = sum(variety_token_counts[v].values())
    func_per_1k = (func_n / total * 1000) if total else 0
    cont_per_1k = (cont_n / total * 1000) if total else 0
    print(f"{v:<10}{func_n:>10} ({func_per_1k:>4.2f}/1k){cont_n:>10} ({cont_per_1k:>4.2f}/1k)")

In [ ]:
print("Function-word hits per variety AFTER cleaning:")
for v in VARIETY_ORDER:
    hits = [(w, variety_token_counts[v].get(w, 0))
            for w in HINDI_FUNCTION_WORDS_CLEAN
            if variety_token_counts[v].get(w, 0) > 0]
    hits.sort(key=lambda x: -x[1])
    print(f"\n{v} (all hits): {hits[:20] if hits else '(none)'}")

In [ ]:
def code_mix_features(text):
    tokens = simple_tokenize(text)
    n_tokens = len(tokens)
    if n_tokens == 0:
        return (0, 0, 0, 0.0, 0.0, False, False)
    func_hits = sum(1 for t in tokens if t in HINDI_FUNCTION_WORDS_CLEAN)
    content_hits = sum(1 for t in tokens if t in INDIC_CONTENT_WORDS)
    any_hits = func_hits + content_hits
    func_rate = func_hits / n_tokens
    total_rate = any_hits / n_tokens
    has_func = func_hits > 0
    has_any = any_hits > 0
    return (n_tokens, func_hits, content_hits, func_rate, total_rate, has_func, has_any)

In [ ]:
# apply to all training examples
feats = train_df['text'].apply(code_mix_features).tolist()
train_df['n_tokens'] = [f[0] for f in feats]
train_df['hindi_func_hits'] = [f[1] for f in feats]
train_df['indic_content_hits'] = [f[2] for f in feats]
train_df['hindi_func_rate'] = [f[3] for f in feats]
train_df['indic_total_rate'] = [f[4] for f in feats]
train_df['has_hindi_func'] = [f[5] for f in feats]
train_df['has_any_indic'] = [f[6] for f in feats]

In [ ]:
# per-variety stats
print("Per-example code mixing statistics per variety (training data):")
stats = train_df.groupby('variety').agg(
    n_examples=('text', 'size'),
    pct_with_hindi_func=('has_hindi_func', lambda x: x.mean() * 100),
    pct_with_any_indic=('has_any_indic', lambda x: x.mean() * 100),
    mean_func_rate=('hindi_func_rate', lambda x: x.mean() * 1000),
    mean_total_rate=('indic_total_rate', lambda x: x.mean() * 1000),).round(2)
stats.columns = ['N', '% w/ Hindi func', '% w/ any Indic', 'Mean func/1k tok', 'Mean total/1k tok']
print(stats.loc[VARIETY_ORDER])

In [ ]:
# focusing on en-IN for the sarcasm correlation
en_in = train_df[train_df['variety'] == 'en-IN']

In [ ]:
# stratifying by whether example has Hindi function words
for flag_col, flag_name in [
    ('has_hindi_func',  'Has Hindi function word'),
    ('has_any_indic',   'Has any Indic marker'),
]:
    print(f"\n-- Stratified by: {flag_name} --")
    tab = en_in.groupby(flag_col).agg(
        N=('Sarcasm', 'size'),
        sarc_count=('Sarcasm', 'sum'),
        sarc_rate=('Sarcasm', 'mean'),
    )
    tab['sarc_rate'] = (tab['sarc_rate'] * 100).round(2)
    tab.index = tab.index.map({False: 'No', True: 'Yes'})
    print(tab)

In [ ]:
# also stratify by source,  Reddit has sarcasm and Reddit may also be where code mixing is more common
print("Code-mixing rate within en-IN by source (Google vs Reddit):")
cm_by_source = en_in.groupby('source').agg(
    N=('text', 'size'),
    pct_hindi_func=('has_hindi_func', lambda x: round(x.mean() * 100, 2)),
    pct_any_indic=('has_any_indic', lambda x: round(x.mean() * 100, 2)),
    sarc_rate=('Sarcasm', lambda x: round(x.mean() * 100, 2)),
)
print(cm_by_source)

In [ ]:
# code-mixing x sarcasm, controlling for source
print("Sarcasm rate by code-mixing status x source, within en-IN:")
crosstab = en_in.groupby(['source', 'has_hindi_func']).agg(N=('Sarcasm', 'size'),sarc_rate=('Sarcasm', lambda x: round(x.mean() * 100, 2))).reset_index()
crosstab['has_hindi_func'] = crosstab['has_hindi_func'].map({False: 'No Hindi', True: 'Hindi'})
print(crosstab.to_string(index=False))

In [ ]:
# 2x2 table within en-IN Reddit: Hindi x Sarcasm
en_in_reddit = en_in[en_in['source'] == 'Reddit']
tab = pd.crosstab(en_in_reddit['has_hindi_func'], en_in_reddit['Sarcasm'])
print("en-IN Reddit contingency table (Hindi function word x Sarcasm):")
print(tab)

In [ ]:
odds_ratio, p_value = fisher_exact(tab.values)
print(f"Fisher's exact test:")
print(f"Odds ratio: {odds_ratio:.2f}")
print(f"p-value:    {p_value:.4f}")

### Observation

1.Domain x variety confound: Most of the sarcasm shows  in Reddit data (e.g. en-UK Google: 0.1% vs en-AU Reddit: 41.8%). Since each variety has a mix of sources (en-AU is 65% Reddit, en-UK is 64% Google), comparing sarcasm across varieties is partly picking up domain effects, not just linguistic differences.

2.Class imbalance differs by variety: Sentiment is fairly even across all three (around 46-52% positive), but sarcasm isnt. Its about 29% in en-AU, compared to  7-8% in en-UK and en-IN. means class weights for sarcasm shouldnt be fixed globally they need to be recalculated for each training.

3.Text length varies by both variety and source:en-IN texts are shorter (median 24 words) than en-AU and en-UK (around 38-46). Reddit comments are shorter than Google reviews. Overall, this is mostly short text data, which likely suits contextual models better than sparse, feature based ones.

4.Linguistic distance is small but noticeable: TF-IDF cosine scores are very high (≥0.947), suggesting all varieties share a strong common core vocabulary. But the filtered Jaccard scores show: en-UK and en-AU are closer to each other (0.484) than either is to en-IN (around 0.40). So the differences are there just more visible in the mid-frequency vocabulary.

5.Hindi grammatical code-mixing in en-IN:About 7.3% of en-IN training examples include at least one Hindi function word, compared to only 0.08-0.17% in en-UK and en-AU. Within en-IN Reddit data, these code-mixed examples are more likely to be sarcastic (20.4% vs 11.7%; Fishers exact test, OR = 1.94, p = 0.029). This points to a reason for weaker transfer.Models trained on inner-circle English wont have seen these Hindi discourse markers, even though they can carry sarcasm cues in en-IN.

## Stage 2: Classical baselines

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.svm import LinearSVC
import time
import json
from sklearn.linear_model import LogisticRegression

In [ ]:
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

In [ ]:
# returns a dict of metrics in a consistent format. Used in all stages so results are comparable
def evaluate_predictions(y_true, y_pred, label_names=None):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    per_class_p, per_class_r, per_class_f1, support = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0, labels=[0, 1])
    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist()
    return {
        'accuracy':      round(float(acc), 4),
        'macro_f1':      round(float(macro_f1), 4),
        'macro_precision': round(float(macro_p), 4),
        'macro_recall':  round(float(macro_r), 4),
        'class_0_precision': round(float(per_class_p[0]), 4),
        'class_0_recall':    round(float(per_class_r[0]), 4),
        'class_0_f1':        round(float(per_class_f1[0]), 4),
        'class_0_support':   int(support[0]),
        'class_1_precision': round(float(per_class_p[1]), 4),
        'class_1_recall':    round(float(per_class_r[1]), 4),
        'class_1_f1':        round(float(per_class_f1[1]), 4),
        'class_1_support':   int(support[1]),
        'confusion_matrix':  cm,   # [[TN, FP], [FN, TP]]
    }

In [ ]:
# catches majority-class failure
def majority_baseline(y_train, y_test):
    majority = int(pd.Series(y_train).mode().iloc[0])
    y_pred = np.full(len(y_test), majority)
    return evaluate_predictions(y_test, y_pred)

In [ ]:
# collects all experimental results across stages
all_results = []

In [ ]:
# appendding a result row so nothing is lost on restart
def log_result(row):
    all_results.append(row)
    with open(RESULTS_DIR / "all_results.json", "w") as f:
        json.dump(all_results, f, indent=2)
print(f"Results will be saved to: {RESULTS_DIR / 'all_results.json'}")

In [ ]:
# run majority baseline for both tasks using existing split
for task in ['Sentiment', 'Sarcasm']:
    print(f"\n-- Task: {task} (all varieties pooled) --")
    y_train = train_df[task].values
    y_test  = test_df[task].values
    metrics = majority_baseline(y_train, y_test)
    print(f"  Accuracy:     {metrics['accuracy']:.4f}")
    print(f"  Macro-F1:     {metrics['macro_f1']:.4f}")
    print(f"  Class 0 F1:   {metrics['class_0_f1']:.4f}  (support={metrics['class_0_support']})")
    print(f"  Class 1 F1:   {metrics['class_1_f1']:.4f}  (support={metrics['class_1_support']})")
    print(f"  Confusion:    {metrics['confusion_matrix']}")
    log_result({'stage': 'Q2.1','task': task,'model': 'majority_baseline','training_data': 'all_varieties_pooled','seed': None,**metrics})
print(f"\nLogged {len(all_results)} results so far.")

In [ ]:
SEEDS = [42, 123]

In [ ]:
# train TF-IDF + Logistic Regression on pooled training data for one task
def run_tfidf_lr(task, seeds=SEEDS, verbose=True):
    X_train_text = train_df['text'].values
    X_test_text  = test_df['text'].values
    y_train = train_df[task].values
    y_test  = test_df[task].values
    # fit on train only
    vectorizer = TfidfVectorizer(ngram_range=(1, 2),min_df=2,max_df=0.95,sublinear_tf=True,lowercase=True,)
    t0 = time.time()
    X_train = vectorizer.fit_transform(X_train_text)
    X_test  = vectorizer.transform(X_test_text)
    vec_time = time.time() - t0
    if verbose:
        print(f"  TF-IDF matrix: train={X_train.shape}, test={X_test.shape}")
        print(f"  Vectorisation time: {vec_time:.2f}s")
    run_metrics = []
    for seed in seeds:
        clf = LogisticRegression(solver='liblinear',class_weight='balanced',max_iter=1000,random_state=seed,C=1.0,)
        t0 = time.time()
        clf.fit(X_train, y_train)
        train_time = time.time() - t0
        t0 = time.time()
        y_pred = clf.predict(X_test)
        inference_time = time.time() - t0
        metrics = evaluate_predictions(y_test, y_pred)
        metrics['train_time_sec'] = round(train_time, 3)
        metrics['inference_time_sec'] = round(inference_time, 3)
        run_metrics.append((seed, metrics))
        if verbose:
            print(f"  seed={seed}: macro-F1={metrics['macro_f1']:.4f} | "
                  f"class1-F1={metrics['class_1_f1']:.4f} | "
                  f"train={train_time:.2f}s")
        log_result({'stage': 'Q2.1','task': task,'model': 'tfidf_logreg','training_data': 'all_varieties_pooled','seed': seed,**metrics})
    # mean +/- std across seeds
    macro_f1s = [m[1]['macro_f1'] for m in run_metrics]
    class1_f1s = [m[1]['class_1_f1'] for m in run_metrics]
    print(f"Mean Macro-F1: {np.mean(macro_f1s):.4f} (+-{np.std(macro_f1s):.4f})")
    print(f"Mean Class-1 F1: {np.mean(class1_f1s):.4f} (+-{np.std(class1_f1s):.4f})")
    return run_metrics

In [ ]:
for task in ['Sentiment', 'Sarcasm']:
    print(f"\n-- Task: {task} --")
    run_tfidf_lr(task)
print(f"\nLogged {len(all_results)} results so far.")

In [ ]:
# TF-IDF + LinearSVM on pooled training data
def run_tfidf_svm(task, seeds=SEEDS, verbose=True):
    X_train_text = train_df['text'].values
    X_test_text  = test_df['text'].values
    y_train = train_df[task].values
    y_test  = test_df[task].values
    vectorizer = TfidfVectorizer(ngram_range=(1, 2),min_df=2,max_df=0.95,sublinear_tf=True,lowercase=True,)
    X_train = vectorizer.fit_transform(X_train_text)
    X_test  = vectorizer.transform(X_test_text)
    for seed in seeds:
        clf = LinearSVC(class_weight='balanced',C=1.0,max_iter=2000,random_state=seed,)
        t0 = time.time()
        clf.fit(X_train, y_train)
        train_time = time.time() - t0
        t0 = time.time()
        y_pred = clf.predict(X_test)
        inference_time = time.time() - t0
        metrics = evaluate_predictions(y_test, y_pred)
        metrics['train_time_sec'] = round(train_time, 3)
        metrics['inference_time_sec'] = round(inference_time, 3)
        if verbose:
            print(f"  seed={seed}: macro-F1={metrics['macro_f1']:.4f} | "
                  f"class1-F1={metrics['class_1_f1']:.4f} | "
                  f"train={train_time:.2f}s")
        log_result({'stage': 'Q2.1','task': task,'model': 'tfidf_svm','training_data': 'all_varieties_pooled','seed': seed,**metrics})

In [ ]:
for task in ['Sentiment', 'Sarcasm']:
    print(f"\n-- Task: {task} --")
    run_tfidf_svm(task)
print(f"\nLogged {len(all_results)} results so far.")

In [ ]:
results_df = pd.DataFrame(all_results)

In [ ]:
# filter to classical baselines only
classical = results_df[results_df['model'].isin(['majority_baseline', 'tfidf_logreg', 'tfidf_svm'])].copy()

In [ ]:
# compact summary
summary_cols = ['accuracy', 'macro_f1', 'class_0_f1', 'class_1_f1']

In [ ]:
# average over seeds (majority baseline has no seeds so it just gets single value)
summary = (classical.groupby(['task', 'model'])[summary_cols].mean().round(4).reset_index())

In [ ]:
# fix ordering
model_order = ['majority_baseline', 'tfidf_logreg', 'tfidf_svm']
summary['model'] = pd.Categorical(summary['model'], categories=model_order, ordered=True)
summary = summary.sort_values(['task', 'model']).reset_index(drop=True)
print(summary.to_string(index=False))

In [ ]:
# save to csv for report
summary.to_csv(RESULTS_DIR / "q2_1_classical_summary.csv", index=False)
print(f"\nSaved: {RESULTS_DIR / 'q2_1_classical_summary.csv'}")

### Stage 2 Observation classical baselines
1.Majority-class baseline establishes the floor:Sentiment Macro-F1 = 0.339 (Class-1 F1 = 0.000); Sarcasm Macro-F1 = 0.462 (Class-1 F1 = 0.000) but accuracy = 0.860. The 86% accuracy in sarcasm with zero minority-class detection is precisely the failure mode that the brief warns against, and Macro-F1 is the primary measure.

2.The best classical model on Sarcasm is TF-IDF + Logistic Regression, which has a higher score than SVM (0.593, 0.285). Sentiment, SVM slightly wins over LR (0.841 vs 0.829).

3.The LR-vs-SVM split on Sarcasm is methodologically informative:The calibrated probabilities offered by LR allow class weighting=balanced to cleanly translate into a shifted decision threshold; the hinge loss offered by SVM adjusts margin penalties without explicitly making a shift in the decision boundary, so SVM under-predicts the minority class (Class-1 F1 0.285 vs LR 0.383) despite identical class weighting.

4.Classical pipelines can be seen as almost deterministic on fixed data:Both seeds (42, 123) give identical Macro-F1 results on four decimal places (std = 0.000 in both classifiers and in both tasks). This meets the requirement of "≥2 runs per setup" but does not give information on seed variance; the information on seed variance will arrive at Stage 3 (transformer fine-tuning).

5.All classical models clear the floor by a big margin:Sentiment Macro-F1 jumps 0.339 (majority) to 0.829-0.841 (classical); Sarcasm between 0.462 and 0.593-0.621. The change in F1 on Sarcasm in class-1 F1 on Sarcasm is 0.285-0.383-0.000, which confirms that the minority class is being detected by the classifiers and not majority prediction (collapses to majority prediction).

## Stage 3: Fine-tuned Transformer

In [ ]:
import torch
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, set_seed
import torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight
from transformers import Trainer
from transformers import DataCollatorWithPadding
from transformers import TrainingArguments
from transformers import AutoModelForSequenceClassification
import shutil
import gc

In [ ]:
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("warning")

In [ ]:
MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Loaded tokenizer: {MODEL_NAME}")
print(f"Model max length: {tokenizer.model_max_length}")

In [ ]:
# build a tokenized HF DatasetDict for one task ready for trainer
def build_hf_dataset(task, max_length):
    # Building HF Datasets from panda df
    # Keeping only columns we need so HF doesnt carry extra into the model
    dd = DatasetDict({'train':      Dataset.from_pandas(train_df[['text', task]].rename(columns={task: 'labels'}), preserve_index=False),'validation': Dataset.from_pandas(val_df[['text', task]].rename(columns={task: 'labels'}), preserve_index=False),'test':       Dataset.from_pandas(test_df[['text', task]].rename(columns={task: 'labels'}), preserve_index=False),})
    def tok(batch):
        return tokenizer(batch['text'], truncation=True, max_length=max_length, padding=False)
    dd = dd.map(tok, batched=True, remove_columns=['text'])
    return dd

In [ ]:
print("\nTokenizing for Sentiment task (max_length=256)...")
sent_ds = build_hf_dataset('Sentiment', max_length=256)
print(sent_ds)

In [ ]:
sarc_ds = build_hf_dataset('Sarcasm', max_length=384)
print(sarc_ds)

In [ ]:
# print one tokenized example
print("Example from sentiment train set:")
print(f"labels: {sent_ds['train'][0]['labels']}")
print(f"n input_ids: {len(sent_ds['train'][0]['input_ids'])}")
print(f"decoded: {tokenizer.decode(sent_ds['train'][0]['input_ids'][:30])}...")

In [ ]:
# class weights from training labels for each task
def get_class_weights(task):
    y = train_df[task].values
    weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=y)
    return torch.tensor(weights, dtype=torch.float)

In [ ]:
cw_sent = get_class_weights('Sentiment')
cw_sarc = get_class_weights('Sarcasm')
print(f"Sentiment class weights: {cw_sent.numpy()}")
print(f"Sarcasm   class weights: {cw_sarc.numpy()}")

In [ ]:
# trainer subclass that applies class weighted cross-entropy loss
class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels").long()          # cast to long
        outputs = model(**inputs)
        logits = outputs.logits
        weights = self.class_weights.to(logits.device)
        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits.view(-1, 2), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [ ]:
# called by HF trainer after each validation epoch
def compute_metrics_hf(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    m = evaluate_predictions(labels, preds)
    return {'macro_f1':   m['macro_f1'],'accuracy':   m['accuracy'],'class_1_f1': m['class_1_f1'],}

In [ ]:
# checkpoints go to scratch
SCRATCH_RUNS = PROJECT_ROOT / "_runs"
SCRATCH_RUNS.mkdir(exist_ok=True)

In [ ]:
# finetune roberta-base on one task with one seed, returns test set metrics dict. cleans up checkpoints before returning
def train_roberta(task, seed, num_epochs=4, lr=2e-5,batch_train=16, batch_eval=32, verbose=True):
    # pick tokenized DatasetDict + class weights for this task
    if task == 'Sentiment':
        ds_task = sent_ds
        class_weights = cw_sent
    else:
        ds_task = sarc_ds
        class_weights = cw_sarc
    set_seed(seed) # sets torch, numpy, random, cuda
    # fresh model instance
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2,id2label={0: 'class_0', 1: 'class_1'},label2id={'class_0': 0, 'class_1': 1},)
    run_dir = SCRATCH_RUNS / f"roberta_{task.lower()}_seed{seed}"
    if run_dir.exists():
        shutil.rmtree(run_dir)
    args = TrainingArguments(output_dir=str(run_dir),num_train_epochs=num_epochs,learning_rate=lr,per_device_train_batch_size=batch_train,per_device_eval_batch_size=batch_eval,warmup_ratio=0.1,
        weight_decay=0.01,logging_steps=50,eval_strategy='epoch',save_strategy='epoch',save_total_limit=1,load_best_model_at_end=True,metric_for_best_model='macro_f1',
        greater_is_better=True,fp16=True,seed=seed,data_seed=seed,report_to='none',disable_tqdm=False,)
    data_collator = DataCollatorWithPadding(tokenizer)
    trainer = WeightedTrainer(class_weights=class_weights,model=model,args=args,train_dataset=ds_task['train'],eval_dataset=ds_task['validation'],
        processing_class=tokenizer,data_collator=data_collator,compute_metrics=compute_metrics_hf,)
    # train
    t0 = time.time()
    trainer.train()
    train_time = time.time() - t0
    # evaluating on test set
    t0 = time.time()
    preds_output = trainer.predict(ds_task['test'])
    inference_time = time.time() - t0
    y_pred = preds_output.predictions.argmax(axis=-1)
    y_true = preds_output.label_ids
    metrics = evaluate_predictions(y_true, y_pred)
    metrics['train_time_sec'] = round(train_time, 2)
    metrics['inference_time_sec'] = round(inference_time, 2)
    if verbose:
        print(f"seed={seed}: macro-F1={metrics['macro_f1']:.4f} | "
              f"class1-F1={metrics['class_1_f1']:.4f} | train={train_time:.1f}s")
    log_result({'stage': 'Q2.1','task': task,'model': 'roberta_base','training_data': 'all_varieties_pooled','seed': seed,**metrics})
    # delete the checkpoint dir and free gpu
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
    if run_dir.exists():
        shutil.rmtree(run_dir)
    return metrics

In [ ]:
print(f"Checkpoints will be written to: {SCRATCH_RUNS}")

In [ ]:
print("RoBERTa-base | Sentiment | seed=42 (pipeline smoke test)")
m = train_roberta('Sentiment', seed=42, num_epochs=4)
print(f"\nFinal test Macro-F1: {m['macro_f1']:.4f}")
print(f"Final test Class-1 F1: {m['class_1_f1']:.4f}")
print(f"Classical-LR baseline was:   Macro-F1 approx 0.829, Class-1 F1 approx 0.821")
print(f"Classical-SVM baseline was:  Macro-F1 approx 0.841, Class-1 F1 approx 0.833")

In [ ]:
print("Sentiment, seed 123")
train_roberta('Sentiment', seed=123, num_epochs=4)
print("Sarcasm, seed 42")
train_roberta('Sarcasm', seed=42, num_epochs=4)
print("Sarcasm, seed 123")
train_roberta('Sarcasm', seed=123, num_epochs=4)
print(f"Total results logged: {len(all_results)}")

In [ ]:
# q2.1 summary
q21 = pd.DataFrame(all_results)
q21 = q21[q21['stage'] == 'Q2.1'].copy()
#aggregated over seeds
summary = (q21.groupby(['task', 'model']).agg(n_runs=('seed', 'size'),macro_f1_mean=('macro_f1', 'mean'),macro_f1_std=('macro_f1', 'std'),class_1_f1_mean=('class_1_f1', 'mean'),class_1_f1_std=('class_1_f1', 'std'),accuracy_mean=('accuracy', 'mean'),).round(4).reset_index())

In [ ]:
# ordering the models
model_order = ['majority_baseline', 'tfidf_logreg', 'tfidf_svm', 'roberta_base']
summary['model'] = pd.Categorical(summary['model'], categories=model_order, ordered=True)
summary = summary.sort_values(['task', 'model']).reset_index(drop=True)

In [ ]:
# std for single run models will be null so replacing with 0
summary['macro_f1_std'] = summary['macro_f1_std'].fillna(0.0)
summary['class_1_f1_std'] = summary['class_1_f1_std'].fillna(0.0)
print(summary.to_string(index=False))
summary.to_csv(RESULTS_DIR / "q2_1_full_summary.csv", index=False)
print(f"\nSaved: {RESULTS_DIR / 'q2_1_full_summary.csv'}")

In [ ]:
for task in ['Sentiment', 'Sarcasm']:
    roberta = summary[(summary['task'] == task) & (summary['model'] == 'roberta_base')]['macro_f1_mean'].iloc[0]
    best_clf_row = (summary[(summary['task'] == task) &(summary['model'].isin(['tfidf_logreg', 'tfidf_svm']))].sort_values('macro_f1_mean', ascending=False).iloc[0])
    best_clf_name = best_clf_row['model']
    best_clf_f1 = best_clf_row['macro_f1_mean']
    gap = roberta - best_clf_f1
    print(f"  {task}: RoBERTa {roberta:.4f} vs best classical ({best_clf_name}) {best_clf_f1:.4f} gap = +{gap:.4f}")

### Stage 3 observation: RoBERTa fine-tuning vs classical baselines

1.RoBERTa-base fine-tuning beats the strongest classical baseline by +0.066 Macro-F1 on Sentiment and +0.063 on Sarcasm. Sentiment final Macro-F1 = 0.907 ± 0.003; Sarcasm final Macro-F1 = 0.709 ± 0.012.

2.The Sarcasm improvement has a stronger qualitative effect than the Sentiment one: Although the absolute Macro-F1 gap is similar across tasks, the per-class breakdown shows the Sarcasm improvement is concentrated in minority-class detection: Class-1 F1 rises by 0.383 (LR) to 0.509 (RoBERTa) - a 26% relative increase. Sentiment, which is already balanced, obtains less per-class.

3.There is 4 times more seed variance on Sarcasm than Sentiment:(±0.012 vs ±0.003 Macro-F1). There is fine-tuning sensitivity to initialisation in the minority-class scarcity, which is reported with mean ± std instead of making the results based on single-seed results.

4.Validation curves confirm the requirement of load_best_model_at-end=True:Sentiment models begin with peak value at epoch 3 and begin to decline by epoch 4 (training loss decreases to around 0.10 and validation Macro-F1 drops by 0.013). Overfitting models of sarcasm are more aggressive - by epoch 4 some seeds have a doubling of validation loss. The non-negotiable design choice is therefore the selection of the validation-best checkpoint by Macro-F1 (instead of the final-epoch model) as a design choice.

5.All transformer training was done using weighted cross-entropy loss:using sklearn-computed inverse-frequency class weights. In the case of Sarcasm, this gave a positive-class weight of an approximation of 3.58 (a 6.15x boost), which is what allows non-zero Class-1 F1 on a 12:1-imbalanced task - the critical difference between this set up and an unweighted majority-collapse baseline.

## Stage 4: Cross-variety evaluation

In [ ]:
import traceback

In [ ]:
# builds a tokenized DatasetDict, train/val from one variety, test from all three
def build_variety_dataset(train_variety, task='Sarcasm', max_length=384):
    tr = train_df[train_df['variety'] == train_variety][['text', task]].rename(columns={task: 'labels'})
    va = val_df[val_df['variety']   == train_variety][['text', task]].rename(columns={task: 'labels'})
    # separate test split per variety so we can evaluate on each
    test_by_variety = {v: test_df[test_df['variety'] == v][['text', task]].rename(columns={task: 'labels'})for v in VARIETY_ORDER}
    train_ds = Dataset.from_pandas(tr, preserve_index=False)
    val_ds   = Dataset.from_pandas(va, preserve_index=False)
    test_ds_by_v = {v: Dataset.from_pandas(df, preserve_index=False)for v, df in test_by_variety.items()}
    def tok(batch):
        return tokenizer(batch['text'], truncation=True, max_length=max_length, padding=False)
    train_ds = train_ds.map(tok, batched=True, remove_columns=['text'])
    val_ds   = val_ds.map(tok, batched=True, remove_columns=['text'])
    test_ds_by_v = {v: d.map(tok, batched=True, remove_columns=['text']) for v, d in test_ds_by_v.items()}
    # class weights from training variety only
    y_tr = np.array(train_ds['labels'])
    weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_tr)
    class_weights_t = torch.tensor(weights, dtype=torch.float)
    # Return as a DatasetDict
    train_val = DatasetDict({'train': train_ds, 'validation': val_ds})
    return train_val, test_ds_by_v, class_weights_t, weights


In [ ]:
# printing to check
for variety in VARIETY_ORDER:
    tv, tbv, cw_t, cw = build_variety_dataset(variety)
    n_sarc = sum(tv['train']['labels'])
    n_tot  = len(tv['train']['labels'])
    print(f"  {variety}: train={n_tot} ({n_sarc} sarcastic, {n_sarc/n_tot*100:.1f}%) "
          f"val={len(tv['validation'])} | class_weights={cw.round(2)}")

In [ ]:
print("VARIETY_ORDER =", VARIETY_ORDER)
print(f"train_df['variety'] unique values: {sorted(train_df['variety'].unique())}")
# run each build independently with try except for failures
for variety in VARIETY_ORDER:
    print(f"\n--- Testing build for {variety} ---")
    try:
        tv, tbv, cw_t, cw = build_variety_dataset(variety)
        n_sarc = sum(tv['train']['labels'])
        n_tot  = len(tv['train']['labels'])
        print(f"  OK: {variety}: train={n_tot} ({n_sarc} sarcastic) val={len(tv['validation'])} class_weights={cw.round(2)}")
    except Exception as e:
        print(f"FAILED on {variety}: {type(e).__name__}: {e}")
        traceback.print_exc()

In [ ]:
# fine-tune on one varietys training data, evaluate on all three test varieties
def train_roberta_crossvariety(train_variety, seed, task='Sarcasm',num_epochs=4, lr=2e-5,batch_train=16, batch_eval=32, verbose=True):
     # Building the variety specific datasets
    max_len = 384  # sarcasm task matches Stage 3 choice
    tv_ds, test_by_v, class_weights_t, _ = build_variety_dataset(train_variety, task=task, max_length=max_len)
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2,id2label={0: 'class_0', 1: 'class_1'},label2id={'class_0': 0, 'class_1': 1},)
    run_dir = SCRATCH_RUNS / f"roberta_cv_{task.lower()}_{train_variety}_seed{seed}"
    if run_dir.exists():
        shutil.rmtree(run_dir)
    args = TrainingArguments(output_dir=str(run_dir),num_train_epochs=num_epochs,learning_rate=lr,per_device_train_batch_size=batch_train,per_device_eval_batch_size=batch_eval,
        warmup_ratio=0.1,weight_decay=0.01,logging_steps=50,eval_strategy='epoch',save_strategy='epoch',save_total_limit=1,load_best_model_at_end=True,
        metric_for_best_model='macro_f1',greater_is_better=True,fp16=True,seed=seed,data_seed=seed,report_to='none',disable_tqdm=False,)
    data_collator = DataCollatorWithPadding(tokenizer)
    trainer = WeightedTrainer(class_weights=class_weights_t,model=model,args=args,train_dataset=tv_ds['train'],eval_dataset=tv_ds['validation'],processing_class=tokenizer,
        data_collator=data_collator,compute_metrics=compute_metrics_hf,)
    t0 = time.time()
    trainer.train()
    train_time = time.time() - t0
    # eval on all 3 test varieties one at a time
    results = []
    for test_v in VARIETY_ORDER:
        t0 = time.time()
        preds_out = trainer.predict(test_by_v[test_v])
        inf_time = time.time() - t0
        y_pred = preds_out.predictions.argmax(axis=-1)
        y_true = preds_out.label_ids
        m = evaluate_predictions(y_true, y_pred)
        m['train_time_sec']     = round(train_time, 2)
        m['inference_time_sec'] = round(inf_time, 2)
        row = {'stage': 'Q2.2','task': task,'model': 'roberta_base_crossvariety','train_variety': train_variety,'test_variety': test_v,'seed': seed,**m}
        log_result(row)
        results.append(row)
        if verbose:
            diag = "in-variety" if test_v == train_variety else ""
            print(f"test={test_v}: macro-F1={m['macro_f1']:.4f} | "
                  f"class1-F1={m['class_1_f1']:.4f}  {diag}")
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
    if run_dir.exists():
        shutil.rmtree(run_dir)
    return results

In [ ]:
for train_variety in VARIETY_ORDER:
    for seed in SEEDS:
        print(f"Train on {train_variety} | seed={seed}")
        train_roberta_crossvariety(train_variety, seed=seed, task='Sarcasm')
print(f"Stage 4 complete. Total results logged: {len(all_results)}")
print(f"Q2.2 entries: {len([r for r in all_results if r['stage'] == 'Q2.2'])}")

In [ ]:
#  putting Q2.2 results into a df
q22 = pd.DataFrame([r for r in all_results if r['stage'] == 'Q2.2'])

In [ ]:
# aggregate over seeds, mean + std per (train, test)
agg = (q22.groupby(['train_variety', 'test_variety']).agg(macro_f1_mean=('macro_f1', 'mean'),macro_f1_std=('macro_f1', 'std'),class_1_f1_mean=('class_1_f1', 'mean'),class_1_f1_std=('class_1_f1', 'std'),class_0_f1_mean=('class_0_f1', 'mean'),accuracy_mean=('accuracy', 'mean'),).round(4).reset_index())


In [ ]:
# reshape to 3x3 matrices
macro_matrix = agg.pivot(index='train_variety', columns='test_variety', values='macro_f1_mean')
macro_matrix = macro_matrix.loc[VARIETY_ORDER, VARIETY_ORDER]
class1_matrix = agg.pivot(index='train_variety', columns='test_variety', values='class_1_f1_mean')
class1_matrix = class1_matrix.loc[VARIETY_ORDER, VARIETY_ORDER]
std_macro_matrix = agg.pivot(index='train_variety', columns='test_variety', values='macro_f1_std')
std_macro_matrix = std_macro_matrix.loc[VARIETY_ORDER, VARIETY_ORDER]

In [ ]:
print("Q2.2 CROSS-VARIETY SARCASM MATRIX (mean Macro-F1 over 2 seeds)")
print(macro_matrix)
print("\n(standard deviation)")
print(std_macro_matrix)
print("CLASS-1 F1 (sarcasm-detection ability specifically)")
print(class1_matrix)

In [ ]:
# diagonal vs off diagonal summary
diag_macro = np.diag(macro_matrix.values).mean()
off_macro  = (macro_matrix.values.sum() - np.diag(macro_matrix.values).sum()) / 6
diag_c1    = np.diag(class1_matrix.values).mean()
off_c1     = (class1_matrix.values.sum() - np.diag(class1_matrix.values).sum()) / 6

In [ ]:
print("VARIETY GAP (in-variety vs cross-variety)")
print(f"Macro-F1:   diagonal={diag_macro:.4f}  off-diagonal={off_macro:.4f}  gap=-{diag_macro-off_macro:.4f}")
print(f"Class-1 F1: diagonal={diag_c1:.4f}  off-diagonal={off_c1:.4f}  gap=-{diag_c1-off_c1:.4f}")

In [ ]:
agg.to_csv(RESULTS_DIR / "q2_2_cross_variety_matrix.csv", index=False)
print(f"Saved: {RESULTS_DIR / 'q2_2_cross_variety_matrix.csv'}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# macro f1 heatmap
ax = axes[0]
sns.heatmap(macro_matrix, annot=True, fmt='.3f', cmap='Blues',vmin=0.45, vmax=0.80, cbar=True, ax=ax,linewidths=1, linecolor='white', square=True,annot_kws={'fontsize': 11, 'fontweight': 'bold'})
# highlight diagonal
for i in range(3):
    ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=False,edgecolor='crimson', lw=3))
ax.set_xlabel("Test variety")
ax.set_ylabel("Train variety")
ax.set_title("Macro-F1 (mean over 2 seeds)")
ax = axes[1]
sns.heatmap(class1_matrix, annot=True, fmt='.3f', cmap='Oranges',vmin=0.10, vmax=0.70, cbar=True, ax=ax,linewidths=1, linecolor='white', square=True,annot_kws={'fontsize': 11, 'fontweight': 'bold'})
for i in range(3):
    ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=False,edgecolor='crimson', lw=3))
ax.set_xlabel("Test variety")
ax.set_ylabel("Train variety")
ax.set_title("Class-1 F1 sarcasm detection only")
plt.suptitle("Cross-variety sarcasm transfer matrix\n"
             "Diagonal (red border) = in-variety training off-diagonal = transfer",fontsize=12, y=1.04)
plt.tight_layout()
save_fig("fig5_crossvariety_matrix")
plt.show()

In [ ]:
# mean of off diagonal entries per row (outgoing) and per column (incoming)
def off_diag_means(matrix):
    arr = matrix.values.copy().astype(float)
    np.fill_diagonal(arr, np.nan)
    row_means = np.nanmean(arr, axis=1) #
    col_means = np.nanmean(arr, axis=0)
    return row_means, col_means

In [ ]:
macro_out, macro_in = off_diag_means(macro_matrix)
class1_out, class1_in = off_diag_means(class1_matrix)

In [ ]:
out_df = pd.DataFrame({'variety': VARIETY_ORDER,'in_variety_c1f1':  np.diag(class1_matrix.values),'outgoing_mean_c1f1': class1_out, 'incoming_mean_c1f1': class1_in,}).round(4)
out_df['out_minus_in_var'] = (out_df['outgoing_mean_c1f1'] - out_df['in_variety_c1f1']).round(4)
print(out_df.to_string(index=False))

In [ ]:
print(f"Best-transferring TRAINING variety (highest outgoing): "
      f"{VARIETY_ORDER[np.argmax(class1_out)]}  (mean off-diag C1-F1 = {class1_out.max():.3f})")
print(f"  Worst-transferring training variety:                   "
      f"{VARIETY_ORDER[np.argmin(class1_out)]}  (mean off-diag C1-F1 = {class1_out.min():.3f})")
print(f"  Most receptive TEST variety (easiest to transfer TO):  "
      f"{VARIETY_ORDER[np.argmax(class1_in)]}  (mean incoming C1-F1 = {class1_in.max():.3f})")
print(f"  Least receptive test variety:                          "
      f"{VARIETY_ORDER[np.argmin(class1_in)]}  (mean incoming C1-F1 = {class1_in.min():.3f})")

In [ ]:
# Answer for report
au_to_uk = class1_matrix.loc['en-AU', 'en-UK']
au_to_in = class1_matrix.loc['en-AU', 'en-IN']
print(f"en-AU en-UK Class-1 F1: {au_to_uk:.4f}")
print(f"en-AU en-IN Class-1 F1: {au_to_in:.4f}")
print(f"Difference: {au_to_uk - au_to_in:+.4f} en-AU transfers "
      f"{'BETTER' if au_to_uk > au_to_in else 'WORSE'} to en-UK than to en-IN.")

### Stage 4 Observation: cross-variety RoBERTa

1.There is a significant variety gap:Mean in-variety Macro-F1 (diagonal) = 0.683 vs cross-variety (off-diagonal) = 0.573, which is a gap of -0.110. This difference is more acute on Class-1 F1 (sarcasm-detection ability specifically): diagonal 0.476 vs off-diagonal 0.300 = -0.176 absolute, a 37% relative decrease.

2.In-variety training wins every row, on both metrics:This confirms that variety-specific features matter and pooled training is not always preferable to per-variety training (although pooled does win in the overall Macro-F1 in the master comparison in Stage 5).

The strongest source variety, 3.en-UK, has a small (0.199) sarcastic training set, which is code-mixing-intensive and hence the weakest.

The most receptive form of target in the case of 4.en-UK is the least receptive target in the case of 4.en-AU:(incoming mean Class-1 F1 = 0.314);en-AU's high rate of sarcasm (41.8% on Reddit) and Australian-specific slang and political terminology make it more difficult to calibrate models trained elsewhere.

5.The explicit question of the brief is answered quantitatively: en-AU transfers better to en-UK by +0.121: en-AU transfers better to en-UK. This ranking follows the Stage 1 Jaccard similarity ranking precisely.

6.Stage 1 Stage 4 closure: the overlap of vocabulary predicts the quality of transfer: Filtered Jaccard rankings (en-UK en-AU 0.484 > en-UK en-IN 0.404 > en-AU en-IN 0.397) equals cross-variety mean Class-1 F1 rankings (0.378 > 0.287 > 0.185). The empirical confirmation of the Q1.2 hypothesis is possible.

7.Patterns of seed-stability are diagnostic of sufficiency of the training signal: the larger training signal admits multiple local optima.

8.The individual epochs displayed Class-1 F1 = 0.000 (full majority-class collapse): it was necessary to verify the value of load-best-model-at-end in order to achieve consistent results.

## Stage 5: LoRA adapters on Gemma-2-2B

In [ ]:
from transformers import BitsAndBytesConfig
from transformers import AutoModelForSequenceClassification
from peft import get_peft_model
from peft import prepare_model_for_kbit_training
from peft import LoraConfig
from peft import TaskType
from peft import PeftModel

In [ ]:
LLM_NAME = "google/gemma-2-2b"

In [ ]:
# 4-bit quantization config (QLoRA)
bnb_config = BitsAndBytesConfig(load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NormalFloat 4-bit this is the best 4-bit format for LLMs
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,     # for extra memory saving and no quality loss
)

In [ ]:
# Gemma specific tokenizer  this is different from RoBERTa
gemma_tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
# Gemma doesnt set a pad token by default we use EOS as a fallback
if gemma_tokenizer.pad_token is None:
    gemma_tokenizer.pad_token = gemma_tokenizer.eos_token

In [ ]:
print(f"Gemma tokenizer loaded. Vocab size: {gemma_tokenizer.vocab_size}")
print(f"Pad token: '{gemma_tokenizer.pad_token}' (id={gemma_tokenizer.pad_token_id})")

In [ ]:
smoke_model = AutoModelForSequenceClassification.from_pretrained(LLM_NAME,num_labels=2,quantization_config=bnb_config,torch_dtype=torch.float16,device_map="auto",)

In [ ]:
# setting pad token id on model config so loss doesnt gets confused during training
smoke_model.config.pad_token_id = gemma_tokenizer.pad_token_id
print(f"loaded successfully")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Model dtype: {smoke_model.dtype}")

In [ ]:
# cleanup
del smoke_model
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after cleanup: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

In [ ]:
# same as build_variety_dataset but using Gemmas tokenizer
def build_gemma_variety_dataset(train_variety, task='Sarcasm', max_length=384):
    tr = train_df[train_df['variety'] == train_variety][['text', task]].rename(columns={task: 'labels'})
    va = val_df[val_df['variety']   == train_variety][['text', task]].rename(columns={task: 'labels'})
    test_by_variety = {v: test_df[test_df['variety'] == v][['text', task]].rename(columns={task: 'labels'})for v in VARIETY_ORDER}
    train_ds = Dataset.from_pandas(tr, preserve_index=False)
    val_ds   = Dataset.from_pandas(va, preserve_index=False)
    test_ds_by_v = {v: Dataset.from_pandas(df, preserve_index=False)for v, df in test_by_variety.items()}
    def tok(batch):
        return gemma_tokenizer(batch['text'], truncation=True, max_length=max_length, padding=False)
    train_ds = train_ds.map(tok, batched=True, remove_columns=['text'])
    val_ds   = val_ds.map(tok, batched=True, remove_columns=['text'])
    test_ds_by_v = {v: d.map(tok, batched=True, remove_columns=['text']) for v, d in test_ds_by_v.items()}
    y_tr = np.array(train_ds['labels'])
    weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_tr)
    class_weights_t = torch.tensor(weights, dtype=torch.float)
    train_val = DatasetDict({'train': train_ds, 'validation': val_ds})
    return train_val, test_ds_by_v, class_weights_t

In [ ]:
# saving lora adapters
ADAPTER_DIR = PROJECT_ROOT / "_adapters"
ADAPTER_DIR.mkdir(exist_ok=True)

In [ ]:
def train_gemma_lora(train_variety, seed, task='Sarcasm',num_epochs=4, lr=2e-4,batch_train=8, batch_eval=16,lora_r=16, lora_alpha=32, lora_dropout=0.05,save_adapter=True,verbose=True):
    tv_ds, test_by_v, class_weights_t = build_gemma_variety_dataset(train_variety, task=task, max_length=384)
    set_seed(seed)
    # loadinfg the 4bit quantised base model
    model = AutoModelForSequenceClassification.from_pretrained(LLM_NAME,num_labels=2,quantization_config=bnb_config,torch_dtype=torch.float16,device_map="auto",)
    model.config.pad_token_id = gemma_tokenizer.pad_token_id
    # preparing for lora, enables gradient checkpointing and makes inputs require gradients
    # so that gradients flow through the frozen quantized weights to the LoRA params
    model = prepare_model_for_kbit_training(model)
    # lora config
    lora_config = LoraConfig(task_type=TaskType.SEQ_CLS,r=lora_r,lora_alpha=lora_alpha,lora_dropout=lora_dropout,bias="none",target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],)
    model = get_peft_model(model, lora_config)
    if verbose:
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in model.parameters())
        print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.3f}%)")
    run_dir = SCRATCH_RUNS / f"gemma_lora_{task.lower()}_{train_variety}_seed{seed}"
    if run_dir.exists():
        shutil.rmtree(run_dir)
    args = TrainingArguments(output_dir=str(run_dir),num_train_epochs=num_epochs,learning_rate=lr,per_device_train_batch_size=batch_train,per_device_eval_batch_size=batch_eval,
        gradient_accumulation_steps=2,warmup_ratio=0.1,weight_decay=0.01,logging_steps=25,eval_strategy='epoch',save_strategy='epoch',save_total_limit=1,
        load_best_model_at_end=True,metric_for_best_model='macro_f1',greater_is_better=True,fp16=True,seed=seed,data_seed=seed,report_to='none',disable_tqdm=False,
        optim="paged_adamw_8bit",)
    data_collator = DataCollatorWithPadding(gemma_tokenizer)
    trainer = WeightedTrainer(class_weights=class_weights_t,model=model,args=args,train_dataset=tv_ds['train'],eval_dataset=tv_ds['validation'],processing_class=gemma_tokenizer,data_collator=data_collator,compute_metrics=compute_metrics_hf,)
    t0 = time.time()
    trainer.train()
    train_time = time.time() - t0
    # eval on all 3 varieties
    results = []
    for test_v in VARIETY_ORDER:
        t0 = time.time()
        preds_out = trainer.predict(test_by_v[test_v])
        inf_time = time.time() - t0
        y_pred = preds_out.predictions.argmax(axis=-1)
        y_true = preds_out.label_ids
        m = evaluate_predictions(y_true, y_pred)
        m['train_time_sec'] = round(train_time, 2)
        m['inference_time_sec'] = round(inf_time, 2)
        row = {'stage': 'Q2.3','task': task,'model': 'gemma2b_lora','train_variety': train_variety,'test_variety': test_v,'seed': seed,**m}
        log_result(row)
        results.append(row)
        if verbose:
            diag = "in-variety" if test_v == train_variety else ""
            print(f"test={test_v}: macro-F1={m['macro_f1']:.4f} | "
                  f"class1-F1={m['class_1_f1']:.4f}  {diag}")
    # saving the adapter, only the LoRA deltas
    if save_adapter:
        adapter_path = ADAPTER_DIR / f"{train_variety}_seed{seed}"
        trainer.model.save_pretrained(str(adapter_path))
        adapter_size_mb = sum(f.stat().st_size for f in adapter_path.rglob('*') if f.is_file()) / 1024**2
        print(f"Adapter saved: {adapter_path} ({adapter_size_mb:.1f} MB)")
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
    if run_dir.exists():
        shutil.rmtree(run_dir)
    return results

In [ ]:
print("Stage 5 LoRA smoke test: Gemma-2-2B on en-UK, seed=42")
results = train_gemma_lora('en-UK', seed=42, num_epochs=4)

In [ ]:
for train_variety in VARIETY_ORDER:
    for seed in SEEDS:
        # skip the en-UK seed=42 run we already completed
        if train_variety == 'en-UK' and seed == 42:
            print(f"skip: {train_variety} seed={seed} (already done)")
            continue
        print(f"Train on {train_variety} | seed={seed}")
        train_gemma_lora(train_variety, seed=seed, task='Sarcasm', num_epochs=4)
print(f"Stage 5 training complete. Total results logged: {len(all_results)}")
print(f"Q2.3 entries: {len([r for r in all_results if r['stage'] == 'Q2.3'])}")

In [ ]:
RESULTS_FILE = PROJECT_ROOT / "results/all_results.json"

In [ ]:
with open(RESULTS_FILE) as f:
    all_results_saved = json.load(f)
print(f"Total results loaded from disk: {len(all_results_saved)}")

In [ ]:
q23 = pd.DataFrame([r for r in all_results_saved if r['stage'] == 'Q2.3'])
print(f"Q2.3 entries:{len(q23)}")
print(f"Train varieties seen:{sorted(q23['train_variety'].unique())}")
print(f"Test  varieties seen:{sorted(q23['test_variety'].unique())}")
print(f"Seeds seen:{sorted(q23['seed'].unique())}")

In [ ]:
# aggregate over seeds
agg = (q23.groupby(['train_variety', 'test_variety']).agg(macro_f1_mean=('macro_f1', 'mean'),macro_f1_std=('macro_f1', 'std'),class_1_f1_mean=('class_1_f1', 'mean'),class_1_f1_std=('class_1_f1', 'std'),class_0_f1_mean=('class_0_f1', 'mean'),accuracy_mean=('accuracy', 'mean'),).round(4).reset_index())

In [ ]:
# 3x3 matrices
VARIETY_ORDER = ['en-UK', 'en-AU', 'en-IN']
macro_lora = agg.pivot(index='train_variety', columns='test_variety', values='macro_f1_mean').loc[VARIETY_ORDER, VARIETY_ORDER]
class1_lora = agg.pivot(index='train_variety', columns='test_variety', values='class_1_f1_mean').loc[VARIETY_ORDER, VARIETY_ORDER]
std_lora = agg.pivot(index='train_variety', columns='test_variety', values='macro_f1_std').loc[VARIETY_ORDER, VARIETY_ORDER]

In [ ]:
print("GEMMA LoRA CROSS-VARIETY MATRIX")
print(macro_lora)
print("(standard deviation)")
print(std_lora)
print("CLASS-1 F1(sarcasm-detection ability)")
print(class1_lora)

In [ ]:
# variety gap summary
diag_macro = np.diag(macro_lora.values).mean()
off_macro  = (macro_lora.values.sum() - np.diag(macro_lora.values).sum()) / 6
diag_c1    = np.diag(class1_lora.values).mean()
off_c1     = (class1_lora.values.sum() - np.diag(class1_lora.values).sum()) / 6

In [ ]:
print("VARIETY GAP Gemma LoRA")
print(f"Macro-F1:diagonal={diag_macro:.4f}  off-diagonal={off_macro:.4f}  gap=-{diag_macro-off_macro:.4f}")
print(f"Class-1 F1:diagonal={diag_c1:.4f}  off-diagonal={off_c1:.4f}  gap=-{diag_c1-off_c1:.4f}")

In [ ]:
OUT_PATH = PROJECT_ROOT / "results/q2_3_lora_matrix.csv"
agg.to_csv(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH}")

In [ ]:
# reload Stage 4 matrix from CSV to compare side by side
Q22_PATH = PROJECT_ROOT / "results/q2_2_cross_variety_matrix.csv"
q22_data = pd.read_csv(Q22_PATH)
macro_roberta = q22_data.pivot(index='train_variety', columns='test_variety',values='macro_f1_mean').loc[VARIETY_ORDER, VARIETY_ORDER]
class1_roberta = q22_data.pivot(index='train_variety', columns='test_variety',values='class_1_f1_mean').loc[VARIETY_ORDER, VARIETY_ORDER]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
# Row 1: Macro-F1
for ax, matrix, title in [(axes[0, 0], macro_roberta, "RoBERTa-base (full fine-tuning)\nMacro-F1"),(axes[0, 1], macro_lora, "Gemma-2-2B + LoRA\nMacro-F1"),]:
    sns.heatmap(matrix, annot=True, fmt='.3f', cmap='Blues',vmin=0.40, vmax=0.80, cbar=True, ax=ax,linewidths=1, linecolor='white', square=True,annot_kws={'fontsize': 11, 'fontweight': 'bold'})
    for i in range(3):
        ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=False,edgecolor='crimson', lw=3))
    ax.set_xlabel("Test variety")
    ax.set_ylabel("Train variety")
    ax.set_title(title, fontsize=11)
# Row 2: Class-1 F1 (sarcasm detection specifically)
for ax, matrix, title in [
    (axes[1, 0], class1_roberta, "RoBERTa-base (full fine-tuning)\nClass-1 F1 (sarcasm detection)"),
    (axes[1, 1], class1_lora,    "Gemma-2-2B + LoRA\nClass-1 F1 (sarcasm detection)"),
]:
    sns.heatmap(matrix, annot=True, fmt='.3f', cmap='Oranges',vmin=0.00, vmax=0.70, cbar=True, ax=ax,linewidths=1, linecolor='white', square=True,annot_kws={'fontsize': 11, 'fontweight': 'bold'})
    for i in range(3):
        ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=False,edgecolor='crimson', lw=3))
    ax.set_xlabel("Test variety")
    ax.set_ylabel("Train variety")
    ax.set_title(title, fontsize=11)
plt.suptitle("Cross-variety sarcasm transfer RoBERTa vs Gemma-2B LoRA\n"
             "Diagonal (red border) = in-variety training  |  2 seeds averaged",fontsize=13, y=1.00)
plt.tight_layout()
FIG_PATH = PROJECT_ROOT / "figures/fig6_lora_vs_roberta_matrix.png"
plt.savefig(FIG_PATH, dpi=200, bbox_inches='tight')
print(f"Saved: {FIG_PATH}")
plt.show()

In [ ]:
def off_diag_means(matrix):
    arr = matrix.values.copy().astype(float)
    np.fill_diagonal(arr, np.nan)
    row_means = np.nanmean(arr, axis=1)
    col_means = np.nanmean(arr, axis=0)
    return row_means, col_means

In [ ]:
macro_out, macro_in  = off_diag_means(macro_lora)
class1_out, class1_in = off_diag_means(class1_lora)

In [ ]:
out_df = pd.DataFrame({'variety': VARIETY_ORDER, 'in_variety_c1f1':    np.diag(class1_lora.values),'outgoing_mean_c1f1': class1_out,'incoming_mean_c1f1': class1_in,}).round(4)
out_df['out_minus_in_var'] = (out_df['outgoing_mean_c1f1'] - out_df['in_variety_c1f1']).round(4)
print(out_df.to_string(index=False))

In [ ]:
print(f"Best-transferring training variety: "
      f"{VARIETY_ORDER[np.argmax(class1_out)]}  (mean off-diag C1-F1 = {class1_out.max():.3f})")
print(f"Worst-transferring training variety: "
      f"{VARIETY_ORDER[np.argmin(class1_out)]}  (mean off-diag C1-F1 = {class1_out.min():.3f})")
print(f"Most receptive test variety (easiest to transfer TO): "
      f"{VARIETY_ORDER[np.argmax(class1_in)]}  (mean incoming C1-F1 = {class1_in.max():.3f})")
print(f"Least receptive test variety: "
      f"{VARIETY_ORDER[np.argmin(class1_in)]}  (mean incoming C1-F1 = {class1_in.min():.3f})")

In [ ]:
au_to_uk = class1_lora.loc['en-AU', 'en-UK']
au_to_in = class1_lora.loc['en-AU', 'en-IN']
print("en-AU en-UK vs en-AU en-IN")
print(f"LoRA en-AU en-UK Class-1 F1: {au_to_uk:.4f}")
print(f"LoRA en-AU en-IN Class-1 F1: {au_to_in:.4f}")
print(f"Difference:{au_to_uk - au_to_in:+.4f} en-AU transfers "
      f"{'BETTER' if au_to_uk > au_to_in else 'WORSE'} to en-UK")

In [ ]:
# Build the Q2 summary for sarcasm
all_q2 = pd.DataFrame([r for r in all_results_saved if r['stage'] in ('Q2.1', 'Q2.2', 'Q2.3')])
sarc_q2 = all_q2[all_q2['task'] == 'Sarcasm'].copy()

In [ ]:
# getting avg metrics for a given (stage, model)
def get_row(stage_df, model_name, split='all'):
    if len(stage_df) == 0:
        return None
    macro = stage_df['macro_f1'].mean()
    macro_std = stage_df['macro_f1'].std() if len(stage_df) > 1 else 0.0
    c1 = stage_df['class_1_f1'].mean()
    c1_std = stage_df['class_1_f1'].std() if len(stage_df) > 1 else 0.0
    return dict(macro_f1_mean=round(macro, 4),macro_f1_std=round(macro_std, 4),class_1_f1_mean=round(c1, 4),class_1_f1_std=round(c1_std, 4),n_runs=len(stage_df))
rows = []

In [ ]:
# Q2.1 pooled training, test set = all varieties merged from majority/LR/SVM/RoBERTa pooled
for model_name in ['majority_baseline', 'tfidf_logreg', 'tfidf_svm', 'roberta_base']:
    sub = sarc_q2[(sarc_q2['stage'] == 'Q2.1') & (sarc_q2['model'] == model_name)]
    r = get_row(sub, model_name)
    if r:
        rows.append({'setup': 'Q2.1 pooled training (test = all)', 'model': model_name, **r})

In [ ]:
# Q2.2: per variety RoBERTa, in variety diagonal only for fair comparison
q22 = sarc_q2[sarc_q2['stage'] == 'Q2.2']
q22_diag = q22[q22['train_variety'] == q22['test_variety']]
r = get_row(q22_diag, 'roberta_base_crossvariety (in-variety)')
rows.append({'setup': 'Q2.2 per-variety RoBERTa (diagonal)',
             'model': 'roberta_base_crossvariety', **r})

In [ ]:
# Q2.2: off diagonal only (cross-variety transfer)
q22_off = q22[q22['train_variety'] != q22['test_variety']]
r = get_row(q22_off, 'roberta_base_crossvariety (off-diagonal)')
rows.append({'setup': 'Q2.2 per-variety RoBERTa (off-diag)','model': 'roberta_base_crossvariety', **r})

In [ ]:
# Q2.3: Gemma LoRA, in-variety diagonal only
q23 = sarc_q2[sarc_q2['stage'] == 'Q2.3']
q23_diag = q23[q23['train_variety'] == q23['test_variety']]
r = get_row(q23_diag, 'gemma2b_lora (in-variety)')
rows.append({'setup': 'Q2.3 Gemma LoRA (diagonal)','model': 'gemma2b_lora', **r})

In [ ]:
# Q2.3: Gemma LoRA off-diagonal
q23_off = q23[q23['train_variety'] != q23['test_variety']]
r = get_row(q23_off, 'gemma2b_lora (off-diagonal)')
rows.append({'setup': 'Q2.3 Gemma LoRA (off-diag)','model': 'gemma2b_lora', **r})
master = pd.DataFrame(rows)

In [ ]:
print("SARCASM TASK across Stages 2.1, 2.2, 2.3")
print(master.to_string(index=False))

In [ ]:
OUT_PATH = PROJECT_ROOT / "results/q2_sarcasm_master_summary.csv"
master.to_csv(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH}")

### Stage 5 summary — Gemma-2-2B LoRA adapters

1.LoRA matches or exceeds full RoBERTa fine-tuning on 2 of 3 in-variety diagonals:despite training only 0.398% of the parameters (6.4M trainable / 1.6B total) and producing 25 MB adapters. en-UK: 0.719 vs 0.702 (+0.017 Macro-F1); en-AU: 0.774 vs 0.752 (+0.022); en-IN: 0.565 vs 0.596 (−0.031).

The regime where the LoRA performs poorly is 2.en-IN:With just 95 sarcastic training examples and heavy code-mixing in the Hindi language, the low-rank constraint is not able to capture the distinctive features of the variety. Class-1 F1 drops from 0.286 (RoBERTa) to 0.167 (LoRA) — a 42% relative decrease. This is an actual architectural constraint: the expressiveness of LoRA is constrained by the sufficiency of the training signals.

3.The most generalist source (0.352 outgoing Class-1 F1) is the so-called best-transferring-type of training, which switches between architectures: In the en-UK case, full fine-tuning is used and the architecture is RoBERTa. In LoRA, the most generalist (0.305) is en-AU. We attribute this to the data-volume sensitivity of LoRA: en-AU has 337 sarcastic training examples which better signal to low-rank features than the 92 of en-UK.

4.The systematically weaker result of LoRA compared to RoBERTa is Mean off-diagonal Class-1 F1 = 0.207 (LoRA) vs 0.300 (RoBERTa), a 31% relative decrease. The most drastic cell -en-IN -en-AU = 0.044 - indicates almost complete failure of transfer. LoRA over-specialises to its training distribution.

5.The trade-off of deployment is favourable despite the cost of transfer: Each LoRA adapter is 25 MB on disk vs a full RoBERTa checkpoint of about 500 MB. To use per-variety deployment (Stage 7B) three LoRA adapters share a single 1.6B-parameter base model in memory and swap in/out in milliseconds. It is the efficiency of deployment advantage the short takes discussion of.

6.Pattern of seed-stability under Stage 4 inverts vs Stage 4: Under RoBERTa, the lowest seed variance (std 0.006) and highest stability were observed with en-IN. en-IN is the most variable (std 0.032) under LoRA. The so-called conservative attractor that en-IN reliably finds under full fine-tuning is not reachable by the low-rank adapter - another indication that the regularisation that en-IN finds under full fine-tuning depends on the sufficiency of the training signals.

7.Master sarcasm summary all Q2 setups (8 rows in Table 4):RoBERTa-base pooled remains the single best sarcasm model (Macro-F1 0.709, Class-1 F1 0.509). Per-variety training (RoBERTa or LoRA) loses 0.02 -0.03 Macro-F1 since more pooled data beats right variety data. The particular benefits of the LoRA, such as small adapters, the ability to swap adapters, the similar in-variety performance, are deployment benefits, but not pure-accuracy benefits.

8.All 6 LoRA adapters saved successfully: (en-UK, en-AU, en-IN × seeds 42, 123) in a directory of: ./_adapters/ and totals 150 MB. The artefacts that will be deployed in Stage 7B are these.

## Stage 6: Evaluation metrics

In [ ]:
RESULTS_PATH = PROJECT_ROOT / "results/all_results.json"
with open(RESULTS_PATH) as f:
    all_results = json.load(f)

In [ ]:
# flatten to DataFrame; fill missing fields with pooled for pooled runs
df = pd.DataFrame(all_results)
df["train_variety"] = df.get("train_variety", pd.Series([None]*len(df))).fillna("pooled")
df["test_variety"]  = df.get("test_variety",  pd.Series([None]*len(df))).fillna("pooled")

In [ ]:
print(f"Flattened:{len(df)} rows")
print(f"Columns:{list(df.columns)}\n")

In [ ]:
# metrics to aggregate across seeds
metric_cols = ["accuracy","macro_precision", "macro_recall", "macro_f1","class_0_precision", "class_0_recall", "class_0_f1","class_1_precision", "class_1_recall", "class_1_f1",]

In [ ]:
# group by config, except seed, and aggregate
group_cols = ["stage", "task", "model", "training_data", "train_variety", "test_variety"]

In [ ]:
agg = (df.groupby(group_cols, dropna=False)[metric_cols].agg(["mean", "std", "count"]).round(4))
agg.columns = [f"{m}_{stat}" for m, stat in agg.columns]
agg = agg.reset_index()
seed_counts = agg["macro_f1_count"].unique()
print(f"Seeds per config:{sorted(seed_counts)}")

In [ ]:
# print sarcasm pooled setups
mask = (df["task"] == "Sarcasm") & (df["training_data"] == "all_varieties_pooled")
sarcasm_pooled = (agg[(agg["task"] == "Sarcasm") & (agg["training_data"] == "all_varieties_pooled")][["model", "macro_f1_mean", "macro_f1_std", "class_1_f1_mean", "class_1_f1_std", "macro_f1_count"]])
print(sarcasm_pooled.to_string(index=False))
print("Sentiment pooled (Q2.1)")
sentiment_pooled = (agg[(agg["task"] == "Sentiment") & (agg["training_data"] == "all_varieties_pooled")][["model", "macro_f1_mean", "macro_f1_std", "class_1_f1_mean", "class_1_f1_std", "macro_f1_count"]])
print(sentiment_pooled.to_string(index=False))


In [ ]:
# save aggregated table
out_path = PROJECT_ROOT / "results/q3_master_aggregated.csv"
agg.to_csv(out_path, index=False)
print(f"Saved master aggregated table {out_path}")
print(f"Shape:{agg.shape[0]} unique configs x {agg.shape[1]} columns")

In [ ]:
# format as mean +/- std or just mean if std is NaN for 1 seed config
def fmt(mean, std, nan_std_as_dash=True):
    if pd.isna(std):
        return f"{mean:.3f}" + ("" if nan_std_as_dash else " +/- --")
    if std == 0.0:
        return f"{mean:.3f} +/- 0.000"
    return f"{mean:.3f} +/- {std:.3f}"

In [ ]:
# Sentiment table (Q2.1 pooled onlf)
sent = agg[(agg["task"] == "Sentiment") & (agg["training_data"] == "all_varieties_pooled")].copy()
model_order_sent = ["majority_baseline", "tfidf_logreg", "tfidf_svm", "roberta_base"]
sent["_ord"] = sent["model"].map({m: i for i, m in enumerate(model_order_sent)})
sent = sent.sort_values("_ord").drop(columns="_ord")
sent_table = pd.DataFrame({
    "Setup": ["Q2.1 Majority baseline","Q2.1 TF-IDF + LogReg","Q2.1 TF-IDF + LinearSVM","Q2.1 RoBERTa-base (pooled)",],
    "Accuracy": [fmt(r.accuracy_mean,r.accuracy_std)for _, r in sent.iterrows()],
    "Macro-P":[fmt(r.macro_precision_mean,r.macro_precision_std)  for _, r in sent.iterrows()],
    "Macro-R":[fmt(r.macro_recall_mean,r.macro_recall_std)for _, r in sent.iterrows()],
    "Macro-F1":[fmt(r.macro_f1_mean,r.macro_f1_std)for _, r in sent.iterrows()],
    "Class-0 F1":[fmt(r.class_0_f1_mean,r.class_0_f1_std)for _, r in sent.iterrows()],
    "Class-1 F1":[fmt(r.class_1_f1_mean,r.class_1_f1_std) for _, r in sent.iterrows()],})
print("SENTIMENT Q3 final evaluation table")
print(sent_table.to_string(index=False))

In [ ]:
# Sarcasm table (Q2.1 pooled + Q2.2/Q2.3 per-variety diag/off-diag)
sarc_pooled = agg[(agg["task"] == "Sarcasm") & (agg["training_data"] == "all_varieties_pooled")].copy()
model_order_sarc = ["majority_baseline", "tfidf_logreg", "tfidf_svm", "roberta_base"]
sarc_pooled["_ord"] = sarc_pooled["model"].map({m: i for i, m in enumerate(model_order_sarc)})
sarc_pooled = sarc_pooled.sort_values("_ord").drop(columns="_ord")

In [ ]:
# helper to get diagonal vs off-diagonal aggregates from raw df
def diag_offdiag_agg(df_raw, task, model_name):
    sub = df_raw[(df_raw["task"] == task) & (df_raw["model"] == model_name)].copy()
    sub["is_diag"] = sub["train_variety"] == sub["test_variety"]
    out = {}
    for diag_flag, label in [(True, "diag"), (False, "offdiag")]:
        s2 = sub[sub["is_diag"] == diag_flag]
        out[label] = {m: (s2[m].mean(), s2[m].std()) for m in metric_cols}
    return out

In [ ]:
rob_cv = diag_offdiag_agg(df,"Sarcasm","roberta_base_crossvariety")
lora= diag_offdiag_agg(df,"Sarcasm","gemma2b_lora")

In [ ]:
def row_from_agg(label, stats):
    return {"Setup":label,"Accuracy":   fmt(*stats["accuracy"]),"Macro-P":fmt(*stats["macro_precision"]),"Macro-R":fmt(*stats["macro_recall"]),
        "Macro-F1":fmt(*stats["macro_f1"]),"Class-0 F1":fmt(*stats["class_0_f1"]),"Class-1 F1":fmt(*stats["class_1_f1"]),}

In [ ]:
sarc_rows = []
sarc_labels = ["Q2.1 Majority baseline","Q2.1 TF-IDF + LogReg","Q2.1 TF-IDF + LinearSVM","Q2.1 RoBERTa-base (pooled)",]

In [ ]:
for label, (_, r) in zip(sarc_labels, sarc_pooled.iterrows()):
    sarc_rows.append({"Setup":label,"Accuracy":fmt(r.accuracy_mean,r.accuracy_std),"Macro-P":fmt(r.macro_precision_mean,r.macro_precision_std),"Macro-R":fmt(r.macro_recall_mean,r.macro_recall_std),"Macro-F1":fmt(r.macro_f1_mean,r.macro_f1_std),"Class-0 F1":fmt(r.class_0_f1_mean,r.class_0_f1_std),"Class-1 F1":fmt(r.class_1_f1_mean,r.class_1_f1_std),})

In [ ]:
sarc_rows.append(row_from_agg("Q2.2 RoBERTa per-variety matched (diag)",rob_cv["diag"]))
sarc_rows.append(row_from_agg("Q2.2 RoBERTa per-variety mismatched (off)",rob_cv["offdiag"]))
sarc_rows.append(row_from_agg("Q2.3 Gemma-2B LoRA matched (diag)",lora["diag"]))
sarc_rows.append(row_from_agg("Q2.3 Gemma-2B LoRA mismatched (off)",lora["offdiag"]))

In [ ]:
sarc_table = pd.DataFrame(sarc_rows)
print("SARCASM Q3 final evaluation table")
print(sarc_table.to_string(index=False))

In [ ]:
sent_out = PROJECT_ROOT / "results/q3_sentiment_final.csv"
sarc_out = PROJECT_ROOT / "results/q3_sarcasm_final.csv"
sent_table.to_csv(sent_out, index=False)
sarc_table.to_csv(sarc_out, index=False)
print(f"Saved:\n  {sent_out}\n  {sarc_out}")

In [ ]:
# average confusion matrix across seeds for pooled configs
def get_mean_confusion(df_raw, task, model_name):
    sub = df_raw[(df_raw["task"] == task)& (df_raw["model"] == model_name)& (df_raw["training_data"] == "all_varieties_pooled")]
    cms = np.stack([np.array(cm) for cm in sub["confusion_matrix"]])
    print(f"{task}/{model_name}:{len(cms)} seed(s)")
    for i, (_, r) in enumerate(sub.iterrows()):
        print(f"seed={r['seed']}  cm={r['confusion_matrix']}  macro_f1={r['macro_f1']:.4f}")
    return cms.mean(axis=0), cms  # mean matrix + stack

In [ ]:
cm_sent_mean,cm_sent_all = get_mean_confusion(df,"Sentiment","roberta_base")
cm_sarc_mean,cm_sarc_all = get_mean_confusion(df,"Sarcasm","roberta_base")

In [ ]:
# plot a confusion matrix with both counts and normalised percentages
def plot_cm(ax, cm, title, class_names):
    cm = np.array(cm)
    row_sums = cm.sum(axis=1, keepdims=True)
    cm_norm = cm / np.where(row_sums == 0, 1, row_sums)
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            count = cm[i, j]
            pct   = cm_norm[i, j] * 100
            txt_color = "white" if cm_norm[i, j] > 0.5 else "black"
            ax.text(j, i, f"{count:.0f}\n({pct:.1f}%)",ha="center", va="center", color=txt_color, fontsize=11)
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names)
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title, fontsize=12)
    return im

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_cm(axes[0], cm_sent_mean,"Sentiment RoBERTa-base (pooled)\nMacro-F1 = 0.902 +/- 0.001",class_names=["Negative", "Positive"],)
plot_cm(axes[1], cm_sarc_mean,"Sarcasm RoBERTa-base (pooled)\nMacro-F1 = 0.709 +/- 0.012",class_names=["Non-sarc", "Sarcastic"],)
fig.suptitle("Figure 7 Confusion matrices for best pooled models (mean over 2 seeds, row-normalised %)",fontsize=13, y=1.02,)
plt.tight_layout()

In [ ]:
out_path = PROJECT_ROOT / "figures/fig7_confusion_matrices.png"
plt.savefig(out_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved{out_path}")

In [ ]:
print("Sentiment (mean over 2 seeds)")
print(f"TN={cm_sent_mean[0,0]:.1f}FP={cm_sent_mean[0,1]:.1f}")
print(f"FN={cm_sent_mean[1,0]:.1f}TP={cm_sent_mean[1,1]:.1f}")
tot = cm_sent_mean.sum()
print(f"Total={tot:.0f} Accuracy={(cm_sent_mean[0,0]+cm_sent_mean[1,1])/tot:.4f}")

In [ ]:
print("Sarcasm (mean over 2 seeds)")
print(f"TN={cm_sarc_mean[0,0]:.1f} FP={cm_sarc_mean[0,1]:.1f}")
print(f"FN={cm_sarc_mean[1,0]:.1f} TP={cm_sarc_mean[1,1]:.1f}")
tot = cm_sarc_mean.sum()
print(f"Total={tot:.0f}  Accuracy={(cm_sarc_mean[0,0]+cm_sarc_mean[1,1])/tot:.4f}")
print(f"Sarcastic class recall: {cm_sarc_mean[1,1]/(cm_sarc_mean[1,0]+cm_sarc_mean[1,1]):.3f}")
print(f"Sarcastic class precision: {cm_sarc_mean[1,1]/(cm_sarc_mean[0,1]+cm_sarc_mean[1,1]):.3f}")

In [ ]:
# a model collapses if it predicts only one class
# we can detect with Class1 F1 = 0 OR Class0 F1 = 0
collapsed = df[(df["class_0_f1"] == 0.0) | (df["class_1_f1"] == 0.0)]

In [ ]:
print(f"Runs with class_0_f1 == 0 OR class_1_f1 == 0:{len(collapsed)}/{len(df)}")
if len(collapsed) > 0:
    print("Collapsed runs:")
    print(collapsed[["task", "model", "train_variety", "test_variety", "seed","class_0_f1", "class_1_f1"]].to_string(index=False))

In [ ]:
# majority baselines are expected to collapse so they are excluded
trained = df[df["model"] != "majority_baseline"].copy()
trained_collapsed = trained[(trained["class_0_f1"] == 0.0) | (trained["class_1_f1"] == 0.0)]
print(f"After excluding majority baselines:")
print(f"Trained runs:{len(trained)}")
print(f"Collapsed trained runs: {len(trained_collapsed)}")

In [ ]:
# printing weakest 10 trained runs to confirm nothing collapsed
print("Weakest 10 trained runs by Class-1 F1 (to confirm all > 0):")
worst = trained.nsmallest(10, "class_1_f1")[["task", "model", "train_variety", "test_variety", "seed","macro_f1", "class_0_f1", "class_1_f1"]]
print(worst.to_string(index=False))

In [ ]:
# summary for report
print("REPORT-READY SUMMARY SENTENCE:")
print(f"""across all {len(trained)} trained runs (4 classical + 4 RoBERTa pooled + 18 RoBERTa
cross-variety + 18 Gemma LoRA cross-variety, 2 seeds each), no model collapsed to
the majority class the minimum observed Class-1 F1 was {trained['class_1_f1'].min():.3f}
and the minimum Class-0 F1 was {trained['class_0_f1'].min():.3f}. by contrast, the
untrained majority baseline achieves Class-1 F1 = 0.000 on both tasks, confirming
that (a) Macro-F1 correctly penalises collapse and (b) every trained model has
learned a non-trivial decision boundary.""")

### Stage 6 Observations

- Most effective models per task:
Sentiment: RoBERTa-base pooled (Macro-F1 0.902 ± 0.001) clear winner - no further optimisation is justified.
Sarcasm overall: RoBERTa-base pooled (Macro-F1 0.709 ± 0.012, Class-1 F1 0.509 ± 0.008).
Sarcasm per-variety (deployable): Gemma + LoRA en-AU seed 123 (Macro-F1 0.793, Class-1 F1 0.709 in-variety).
-Sentiment confusion matrix is symmetric:90% recall on each class, errors are evenly distributed. Healthy balanced classifier.
-Sarcasm confusion matrix is not symmetrical:89% non-sarcastic recall and 57% sarcastic recall. Even when the class weights are used, the pooled model still fails to capture 43% of actual sarcasm.Sarcasm is a structurally more difficult task - pragmatic, not lexical.
-Non-collapse verified: in 48 trained runs, the minimum Class-1 F1 is 0.010 ( vs 0.000 in the untrained majority baseline). Each of the models has been trained on a non-trivial decision boundary.
-Sarcasm precision and recall are both around 0.5: to have the best pooled model, you have to be certain that the model is genuinely uncertain on hard cases, and not systematically biased. This inspires the Stage 7A error analysis.

### Stage 6 Observations

- Most effective models per task:
Sentiment: RoBERTa-base pooled (Macro-F1 0.902 ± 0.001) clear winner - no further optimisation is justified.
Sarcasm overall: RoBERTa-base pooled (Macro-F1 0.709 ± 0.012, Class-1 F1 0.509 ± 0.008).
Sarcasm per-variety (deployable): Gemma + LoRA en-AU seed 123 (Macro-F1 0.793, Class-1 F1 0.709 in-variety).
-Sentiment confusion matrix is symmetric:90% recall on each class, errors are evenly distributed. Healthy balanced classifier.
-Sarcasm confusion matrix is not symmetrical:89% non-sarcastic recall and 57% sarcastic recall. Even when the class weights are used, the pooled model still fails to capture 43% of actual sarcasm.Sarcasm is a structurally more difficult task - pragmatic, not lexical.
-Non-collapse verified: in 48 trained runs, the minimum Class-1 F1 is 0.010 ( vs 0.000 in the untrained majority baseline). Each of the models has been trained on a non-trivial decision boundary.
-Sarcasm precision and recall are both around 0.5: to have the best pooled model, you have to be certain that the model is genuinely uncertain on hard cases, and not systematically biased. This inspires the Stage 7A error analysis.

## Stage 7A: Error analysis and few-shot prompting

In [ ]:
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding
from peft import PeftModel
from torch.utils.data import DataLoader

In [ ]:
# diagonal runs (matched train/test), finding the best adapter benchmark for Stage 7a
lora_runs = df[df["model"] == "gemma2b_lora"].copy()
lora_diag = lora_runs[lora_runs["train_variety"] == lora_runs["test_variety"]].copy()

In [ ]:
print("All 6 diagonal LoRA runs (in-variety performance):")
view_cols = ["train_variety", "seed", "macro_f1", "class_0_f1", "class_1_f1","class_1_precision", "class_1_recall", "accuracy"]
print(lora_diag[view_cols].sort_values(["train_variety", "seed"]).to_string(index=False))

In [ ]:
print("Best LoRA adapter under different criteria:")
best_c1 = lora_diag.loc[lora_diag["class_1_f1"].idxmax()]
best_macro = lora_diag.loc[lora_diag["macro_f1"].idxmax()]
print(f"Best Class-1 F1:{best_c1['train_variety']} seed={int(best_c1['seed'])}")
print(f"Class-1 F1 ={best_c1['class_1_f1']:.4f}  "
      f"Macro-F1 ={best_c1['macro_f1']:.4f}  "
      f"C1-Precision ={best_c1['class_1_precision']:.3f}  "
      f"C1-Recall ={best_c1['class_1_recall']:.3f}")
print(f"Best Macro-F1:    {best_macro['train_variety']} seed={int(best_macro['seed'])}")
print(f"Class-1 F1 = {best_macro['class_1_f1']:.4f}  "
      f"Macro-F1 = {best_macro['macro_f1']:.4f}  "
      f"C1-Precision = {best_macro['class_1_precision']:.3f}  "
      f"C1-Recall = {best_macro['class_1_recall']:.3f}")

In [ ]:
# check adapter path exists on disk
from pathlib import Path
adapters_dir = PROJECT_ROOT / "_adapters"
print("Adapters on disk:")
for p in sorted(adapters_dir.iterdir()):
    if p.is_dir():
        size_mb = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e6
        print(f"  {p.name:25s}  {size_mb:6.1f} MB")

In [ ]:
#config
TARGET_ADAPTER = "en-AU_seed123"
adapter_path = ADAPTER_DIR / TARGET_ADAPTER
print(f"Loading adapter from:{adapter_path}")
assert adapter_path.exists(), f"Adapter not found: {adapter_path}"

In [ ]:
# loading base model in 4 bit
print(f"Loading Gemma-2-2B base in 4-bit...")
t0 = time.time()
base_model = AutoModelForSequenceClassification.from_pretrained(LLM_NAME,num_labels=2,quantization_config=bnb_config,torch_dtype=torch.float16,device_map="auto",)

In [ ]:
base_model.config.pad_token_id = gemma_tokenizer.pad_token_id
print(f"loaded in {time.time()-t0:.1f}s")
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

In [ ]:
print(f"Attaching adapter{TARGET_ADAPTER}...")
model = PeftModel.from_pretrained(base_model, str(adapter_path))
model.eval()
print(f"adapter attached  |  VRAM:{torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
print(f"Tokenizing {len(en_au_test)} en-AU test examples...")
def tok_batch(batch):
    return gemma_tokenizer(batch["text"], truncation=True, max_length=384, padding=False)

In [ ]:
en_au_ds = Dataset.from_pandas(en_au_test[["text", "Sarcasm"]].rename(columns={"Sarcasm": "labels"}),preserve_index=False,)
en_au_ds_tok = en_au_ds.map(tok_batch, batched=True, remove_columns=["text"])
# keep original text for the error analysis
texts = en_au_test["text"].tolist()
true_labels = en_au_test["Sarcasm"].tolist()

In [ ]:
# batched inference
collator = DataCollatorWithPadding(gemma_tokenizer)
loader = DataLoader(en_au_ds_tok, batch_size=16, collate_fn=collator, shuffle=False)
all_logits = []
t0 = time.time()

In [ ]:
with torch.no_grad():
    for i, batch in enumerate(loader):
        batch = {k: v.to(model.device) for k, v in batch.items() if k != "labels"}
        out = model(**batch)
        all_logits.append(out.logits.detach().cpu().float())
        if (i + 1) % 10 == 0:
            print(f"batch {i+1}/{len(loader)}  ({time.time()-t0:.1f}s elapsed)")

In [ ]:
logits = torch.cat(all_logits, dim=0).numpy()
preds = logits.argmax(axis=-1)
probs_sarcastic = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
print(f"inference done in {time.time()-t0:.1f}s")
print(f"predictions shape: {preds.shape}")

In [ ]:
# build results dataframe
results_df = pd.DataFrame({"idx":range(len(texts)),"text":texts,"true_label":true_labels,"pred_label":preds,"prob_sarc":probs_sarcastic.round(4),"correct":np.array(true_labels) == preds,})
results_df["error_type"] = np.where(results_df["correct"], "correct",np.where((results_df["true_label"] == 1) & (results_df["pred_label"] == 0), "FN","FP"))

In [ ]:
# verify metrics match stage 5 result
print("SANITY CHECK en-AU seed 123 LoRA on en-AU test")
print(classification_report(true_labels, preds, target_names=["non-sarc", "sarcastic"], digits=4))
print(f"Macro-F1:{f1_score(true_labels, preds, average='macro'):.4f}")
print(f"Class-1 F1:{f1_score(true_labels, preds, pos_label=1):.4f}")
print(f"Logged in Stage 5: Macro-F1=0.7932, Class-1 F1=0.7089")
print("Confusion matrix [[TN,FP],[FN,TP]]:")
print(confusion_matrix(true_labels, preds))

In [ ]:
preds_path = PROJECT_ROOT / "results" / "q4_enAU_seed123_predictions.csv"
results_df.to_csv(preds_path, index=False)
print(f"Saved {preds_path}")
print(f"Errors: {(~results_df['correct']).sum()}/{len(results_df)} "
      f"({(~results_df['correct']).mean()*100:.1f}%)")
print(f"FN (missed sarcasm):{(results_df['error_type']=='FN').sum()}")
print(f"FP (false alarm):{(results_df['error_type']=='FP').sum()}")

In [ ]:
# extracting 10 stratified errors for linguistic analysis
errors = results_df[~results_df["correct"]].copy()
errors["wrong_confidence"] = np.where(errors["error_type"] == "FN",1 - errors["prob_sarc"],errors["prob_sarc"],)

In [ ]:
# add text length for stratification
errors["word_count"] = errors["text"].str.split().str.len()
fn_errors = errors[errors["error_type"] == "FN"].sort_values("wrong_confidence", ascending=False).head(5).copy()
fp_errors = errors[errors["error_type"] == "FP"].sort_values("wrong_confidence", ascending=False).head(5).copy()

In [ ]:
selected = pd.concat([fn_errors, fp_errors]).reset_index(drop=True)
selected["q4_id"] = range(1, 11)  # ids 1-10 for easy referencing in writeup

In [ ]:
for _, row in selected.iterrows():
    label_word  = "SARCASTIC" if row["true_label"] == 1 else "NOT sarcastic"
    pred_word   = "SARCASTIC" if row["pred_label"] == 1 else "NOT sarcastic"
    err_tag     = "FN (missed)" if row["error_type"] == "FN" else "FP (false alarm)"
    print(f"{row['q4_id']}  [{err_tag}]  model P(sarc)={row['prob_sarc']:.3f}"
          f"({row['word_count']} words)")
    print(f"True: {label_word}   Predicted: {pred_word}")
    print(f"Text: {row['text']}")

In [ ]:
# saving for reference during report
q4_errors_path = PROJECT_ROOT / "results" / "q4_selected_errors.csv"
selected[["q4_id", "idx", "error_type", "true_label", "pred_label","prob_sarc", "wrong_confidence", "word_count", "text"]].to_csv(q4_errors_path, index=False)

In [ ]:
print(f"Saved: {q4_errors_path}")

In [ ]:
# detach the PEFT wrapper without removing gemma model
del model
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after detaching adapter: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
del base_model
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after clearing classification model: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
t0 = time.time()
gen_model = AutoModelForCausalLM.from_pretrained(LLM_NAME,quantization_config=bnb_config,torch_dtype=torch.float16,device_map="auto",)

In [ ]:
gen_model.eval()
print(f"loaded in{time.time()-t0:.1f}s | VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# my written explanations from the error analysis above, feeding into the prompt
few_shot_examples = [
    {
        "text": "It's great, barely any customers and the cinemas are always empty.",
        "label": "Sarcastic",
        "explanation": (
            "The word 'great' sounds positive, but the rest of the sentence "
            "describes a bad situation (empty cinemas = poor business). The "
            "positive word clashes with the negative reality, which is irony."
        ),
    },
    {
        "text": "One ball, one bounce. Thems are the rules.",
        "label": "Not sarcastic",
        "explanation": (
            "The speaker is just stating a rule in informal Australian English. "
            "The word 'thems' is colloquial dialect, not ironic mock-formality. "
            "The statement is sincere."
        ),
    },
    {
        "text": "Cool, without knowing your prospective field, this is impossible to answer",
        "label": "Sarcastic",
        "explanation": (
            "'Cool' here is not genuinely positive. It is a dismissive discourse "
            "marker, and the abrupt shift to the negative assessment "
            "('impossible to answer') shows the speaker is brushing off the "
            "question rather than agreeing with it."
        ),
    },
    {
        "text": "When will the internet stop being racist? Is that really your question?",
        "label": "Not sarcastic",
        "explanation": (
            "This is a rhetorical question used to push back against the original "
            "question's premise. The speaker disagrees genuinely -- they are not "
            "pretending to agree. Rhetorical incredulity is not sarcasm."
        ),
    },
]

In [ ]:
# remaining 6 test examples (not used for few shot)
test_examples = selected[selected["q4_id"].isin([2, 3, 4, 6, 8, 10])].copy()
print(f"Test examples: {len(test_examples)}  (ids {list(test_examples['q4_id'])})")

In [ ]:
# builds a prompt with labelled demos + the target text to classify
def build_fewshot_prompt(shot_examples, target_text):
    parts = [
        "Classify the following text as 'Sarcastic' or 'Not sarcastic'. "
        "A sarcastic statement says the opposite of what the speaker means, often "
        "using positive words about negative situations. Not every informal, "
        "colloquial, or rhetorical statement is sarcastic.\n"
    ]
    for ex in shot_examples:
        parts.append(f"Text:{ex['text']}")
        parts.append(f"Label:{ex['label']}")
        parts.append(f"Reasoning:{ex['explanation']}")
        parts.append("")  # blank line between examples
    parts.append(f"Text: {target_text}")
    parts.append("Label:")
    return "\n".join(parts)

In [ ]:
# preview on example 2, to see what the model receives
preview_prompt = build_fewshot_prompt(few_shot_examples, test_examples.iloc[0]["text"])
print(preview_prompt)
print(f"Prompt length:{len(gemma_tokenizer.encode(preview_prompt))} tokens")

In [ ]:
fewshot_results = []
for _, row in test_examples.iterrows():
    q4_id      = int(row["q4_id"])
    text       = row["text"]
    true_label = int(row["true_label"])
    adapter_pred = int(row["pred_label"])
    prompt = build_fewshot_prompt(few_shot_examples, text)
    inputs = gemma_tokenizer(prompt, return_tensors="pt").to(gen_model.device)
    with torch.no_grad():
        out = gen_model.generate(**inputs,max_new_tokens=10,do_sample=False,temperature=1.0,pad_token_id=gemma_tokenizer.pad_token_id,)
    # decode only the newly generated tokens
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    generated_text = gemma_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    # check for Not sarcastic before Sarcastic, Sarcastic is a substring of Not sarcastic
    low = generated_text.lower()
    if "not sarcastic" in low or low.startswith("not"):
        fewshot_pred = 0
        parse_ok = True
    elif "sarcastic" in low:
        fewshot_pred = 1
        parse_ok = True
    else:
        fewshot_pred = -1  # couldnt parse
        parse_ok = False
    fewshot_results.append({"q4_id":q4_id,"text_preview":text[:80] + ("..." if len(text) > 80 else ""),"true_label":true_label,"adapter_pred":adapter_pred,
        "fewshot_gen":generated_text,"fewshot_pred":fewshot_pred,"adapter_correct":adapter_pred == true_label,"fewshot_correct":fewshot_pred == true_label,"parse_ok":parse_ok,})

In [ ]:
print(f"{'ID':>3}  {'TRUE':<14} {'ADAPTER':<14} {'FEW-SHOT':<14} {'FIXED?':<8} {'GEN OUT'}")
for r in fewshot_results:
    true_w    = "Sarcastic" if r["true_label"] == 1 else "Not sarc"
    adapter_w = "Sarcastic" if r["adapter_pred"] == 1 else "Not sarc"
    if r["fewshot_pred"] == 1:
        fs_w = "Sarcastic"
    elif r["fewshot_pred"] == 0:
        fs_w = "Not sarc"
    else:
        fs_w = "UNPARSED"
    fixed = "FIXED" if r["fewshot_correct"] else "still wrong"
    print(f"{r['q4_id']:>3}  {true_w:<14} {adapter_w:<14} {fs_w:<14} {fixed:<8} {r['fewshot_gen'][:40]!r}")

In [ ]:
total        = len(fewshot_results)
fs_fixed     = sum(r["fewshot_correct"] for r in fewshot_results)
parse_fails  = sum(not r["parse_ok"] for r in fewshot_results)

In [ ]:
print(f"Total test examples:{total}")
print(f"Adapter correct: 0/{total}")
print(f"Base Gemma + few-shot prompt correct:{fs_fixed}/{total}")
print(f"Parsing failures:{parse_fails}/{total}")

In [ ]:
fewshot_out = PROJECT_ROOT / "results" / "q4_fewshot_results.json"
with open(fewshot_out, "w") as f:
    _json.dump(fewshot_results, f, indent=2)
print(f"Saved: {fewshot_out}")

In [ ]:
zero_shot_results = []
zero_shot_instruction = ("Classify the following text as 'Sarcastic' or 'Not sarcastic'")

In [ ]:
for _, row in test_examples.iterrows():
    q4_id      = int(row["q4_id"])
    text       = row["text"]
    true_label = int(row["true_label"])
    prompt = zero_shot_instruction + f"Text: {text}\nLabel:"
    inputs = gemma_tokenizer(prompt, return_tensors="pt").to(gen_model.device)
    with torch.no_grad():out = gen_model.generate(**inputs,max_new_tokens=10,do_sample=False,temperature=1.0,pad_token_id=gemma_tokenizer.pad_token_id,)
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    gen_text = gemma_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    low = gen_text.lower()
    if "not sarcastic" in low or low.startswith("not"):
        zs_pred = 0
    elif "sarcastic" in low:
        zs_pred = 1
    else:
        zs_pred = -1
    zero_shot_results.append({
        "q4_id": q4_id, "true_label": true_label,
        "zeroshot_pred": zs_pred, "zeroshot_gen": gen_text,
    })

In [ ]:
print(f"{'ID':>3}  {'TRUE':<12} {'ZERO-SHOT':<12} {'GEN OUT'}")
for r in zero_shot_results:
    t = "Sarcastic" if r["true_label"] == 1 else "Not sarc"
    if r["zeroshot_pred"] == 1:   z = "Sarcastic"
    elif r["zeroshot_pred"] == 0: z = "Not sarc"
    else:z = "UNPARSED"
    print(f"{r['q4_id']:>3}  {t:<12} {z:<12} {r['zeroshot_gen'][:40]!r}")

In [ ]:
# to compare predictions across the two conditions
zs_preds = [r["zeroshot_pred"] for r in zero_shot_results]
fs_preds = [r["fewshot_pred"]  for r in fewshot_results]
print(f"Zero-shot predictions:{zs_preds}")
print(f"Few-shot predictions:{fs_preds}")
print(f"Agreement:{sum(z==f for z,f in zip(zs_preds, fs_preds))}/6")

In [ ]:
# Q4 summary table
q4_summary = []
for fs_r, zs_r in zip(fewshot_results, zero_shot_results):
    assert fs_r["q4_id"] == zs_r["q4_id"]
    q4_summary.append({"q4_id":fs_r["q4_id"],"true_label":fs_r["true_label"],"text_preview":fs_r["text_preview"],"adapter_pred":fs_r["adapter_pred"],"zeroshot_pred":zs_r["zeroshot_pred"],"fewshot_pred":fs_r["fewshot_pred"],"adapter_correct":fs_r["adapter_pred"]  == fs_r["true_label"],"zeroshot_correct":zs_r["zeroshot_pred"] == fs_r["true_label"],"fewshot_correct":fs_r["fewshot_pred"]  == fs_r["true_label"],})
q4_df = pd.DataFrame(q4_summary)

In [ ]:
def label_str(v):
    return "Sarc" if v == 1 else ("Not-sarc" if v == 0 else "UNPARSED")
def tick(b):
    return "true" if b else "false"

In [ ]:
print(f"{'ID':>3} {'TRUE':<9} {'ADAPTER':<9} {'OK':<3} {'ZERO-SHOT':<9} {'OK':<3} "
      f"{'FEW-SHOT':<9} {'OK':<3}  TEXT")
for _, r in q4_df.iterrows():
    print(
        f"{r['q4_id']:>3}"
        f"{label_str(r['true_label']):<9}  "
        f"{label_str(r['adapter_pred']):<9} {tick(r['adapter_correct']):<3} "
        f"{label_str(r['zeroshot_pred']):<9} {tick(r['zeroshot_correct']):<3} "
        f"{label_str(r['fewshot_pred']):<9} {tick(r['fewshot_correct']):<3}  "
        f"{r['text_preview']}"
    )

In [ ]:
# aggregate counts
print(f"{'CONDITION':<30} {'CORRECT':<10} NOTES")
print(f"{'LoRA adapter (en-AU s123)':<30} {q4_df['adapter_correct'].sum():>3}/6    "
      f"(all 6 are known errors by construction)")
print(f"{'Base Gemma zero-shot':<30} {q4_df['zeroshot_correct'].sum():>3}/6    "
      f"(predicts Not sarcastic by default)")
print(f"{'Base Gemma few-shot (4 demos)':<30} {q4_df['fewshot_correct'].sum():>3}/ 6    "
      f"(predicts Sarcastic on all inputs)")

In [ ]:
q4_df.to_csv(PROJECT_ROOT / "results" / "q4_final_comparison.csv", index=False)
print(f"Saved: {PROJECT_ROOT / 'results' / 'q4_final_comparison.csv'}")

### Stage 7A: Observations
-Selected adapter: en-AU seed 123 LoRA (best in-variety adapter, Macro-F1 0.793, Class-1 F1 0.709). Confusion matrix demonstrates a balanced minority profile: 87.5% non-sarc recall, 71.4% sarc recall, 70.4% sarc precision.
-Error selection methodology: we selected the 5 most confidently wrong examples in each direction out of 115 errors (56 FN + 59 FP). All 10 chosen errors had wrong-class confidence ≈ 1.000 - they are confident failures in reasoning, not borderline failures of misclassification.
-Linguistic failure modes (experimentally determined by manual analysis of 4 examples):
  1. Missing irony in absence of explicit marker of sarcasm (e.g., It's great, [empty cinemas])
  2. Confusion of informal/dialectal register and sarcasm (e.g., Thems are the rules)
  3. Confusion of rhetorical and sarcastic agreement.
- Few-shot vs zero-shot vs adapter: on 6 held-out errors:
  Adapter: 0/6 correct (by construction - these are errors of the adapters)
  Zero-shot (instruction only): 3/6 correct, but fails on 4/6.
  Few-shot (4 demonstrations): 3/6 correct, but predicts Sarcastic, on 6/6 - collapses to a single output.
-The 3/6 = 3/6 finding is not accurate:both prompt conditions fail to do the task in their own ways. The correct disjoint (zero-shot acquires the FPs right, few-shot acquires the FNs right) and confirming that neither of the two acquires the task.
-Headline result: prompt engineering on a non-instruction-tuned base model does not constitute an alternative to parameter-efficient fine-tuning. The behaviour that 337 sarcastic training instances in a 25 MB adapter cause cannot be induced by four well-chosen, linguistically-motivated demonstrations.

## Stage 7B:Deployment

In [ ]:
from huggingface_hub import HfApi, create_repo, whoami
from pathlib import Path
import subprocess

In [ ]:
# uploading the 3 best adapters to HF hub
api = HfApi()
user_info = whoami()
HF_USER = user_info["name"]
print(f"Logged in as: {HF_USER}")

In [ ]:
# seed 123 across all varieties, best within-variety performers
ADAPTER_DIR = PROJECT_ROOT / "_adapters"
adapters_to_upload = [{"variety": "en-UK", "local_path": ADAPTER_DIR / "en-UK_seed123"},{"variety": "en-AU", "local_path": ADAPTER_DIR / "en-AU_seed123"},{"variety": "en-IN", "local_path": ADAPTER_DIR / "en-IN_seed123"},]

In [ ]:
# README template, the model card on HF hub
def make_readme(variety, macro_f1, class1_f1):
    return f"""---
base_model: google/gemma-2-2b
library_name: peft
tags:
- lora
- gemma
- sarcasm-detection
- besstie
language:
- en
---
# BESSTIE Sarcasm LoRA Adapter -- {variety}
LoRA adapter for sarcasm detection on **{variety}** English variety, fine-tuned on the BESSTIE dataset (Surrey NLP).
## Performance (in-variety test set)
- Macro-F1: {macro_f1:.4f}
- Class-1 (sarcastic) F1: {class1_f1:.4f}
## Usage
```python
from transformers import AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
)
base = AutoModelForSequenceClassification.from_pretrained(
    "google/gemma-2-2b", num_labels=2,
    quantization_config=bnb_config, torch_dtype=torch.float16, device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b")
base.config.pad_token_id = tokenizer.pad_token_id
model = PeftModel.from_pretrained(base, "{HF_USER}/besstie-sarcasm-gemma2b-lora-{variety.lower()}")
```
## Training details
- LoRA r=16, alpha=32, dropout=0.05, target modules q_proj/k_proj/v_proj/o_proj
- 4-bit NF4 quantization with double-quant
- 4 epochs, lr=2e-4, paged_adamw_8bit
- Class-weighted cross-entropy
## Coursework context
Trained for COMM061 NLP coursework (University of Surrey, 2026) using the
[surrey-nlp/BESSTIE-CW-26](https://huggingface.co/datasets/surrey-nlp/BESSTIE-CW-26) dataset.
"""

In [ ]:
# get per-variety metrics from the aggregated table for the README
# (we use the diagonal lora seed 123 metrics, these are the in-variety scores)
df_perf = pd.DataFrame(all_results)
def get_metrics(variety):
    row = df_perf[(df_perf["model"] == "gemma2b_lora")& (df_perf["train_variety"] == variety)& (df_perf["test_variety"]  == variety)& (df_perf["seed"] == 123)].iloc[0]
    return float(row["macro_f1"]), float(row["class_1_f1"])

In [ ]:
# upload loop
for ad in adapters_to_upload:
    variety= ad["variety"]
    local= ad["local_path"]
    repo_name=f"besstie-sarcasm-gemma2b-lora-{variety.lower()}"
    repo_id=f"{HF_USER}/{repo_name}"
    print(f"{variety}:uploading to {repo_id}")
    print(f"local:{local}")
    print(f"files:{[p.name for p in local.iterdir() if p.is_file()]}")
    create_repo(repo_id, exist_ok=True, private=False)
    macro_f1, c1_f1 = get_metrics(variety)
    readme_path = local / "README.md"
    readme_path.write_text(make_readme(variety, macro_f1, c1_f1))
    print(f"README written ({readme_path.stat().st_size} bytes) -- Macro-F1={macro_f1:.4f}, C1-F1={c1_f1:.4f}")
    api.upload_folder(folder_path=str(local),repo_id=repo_id,commit_message=f"Upload BESSTIE sarcasm adapter ({variety}, seed 123)",)
    print(f"uploaded: https://huggingface.co/{repo_id}")

In [ ]:
#write app.py
APP_PATH = PROJECT_ROOT / "app.py"
APP_CODE = '''"""
BESSTIE Sarcasm Detector — Gradio deployment
A web UI for sarcasm detection across British (en-UK), Australian (en-AU),
and Indian (en-IN) English varieties. Built on Gemma-2-2B with three
variety-specific LoRA adapters served from HuggingFace Hub.
Usage:
    python app.py
The app loads the base model once at startup, then registers all three
adapters by name. Variety switches at request time are O(0.01s) — the
LoRA "ship tiny adapters, swap cheaply" deployment story in practice.
COMM061 NLP coursework, University of Surrey, 2026.
"""
import time
import torch
import gradio as gr
from transformers import (AutoModelForSequenceClassification,AutoTokenizer,BitsAndBytesConfig,)
from peft import PeftModel
#configuration
BASE_MODEL = "google/gemma-2-2b"
HF_USER    = "Mohammadeeu20"
ADAPTER_REPOS = {"British (en-UK)":   f"{HF_USER}/besstie-sarcasm-gemma2b-lora-en-uk","Australian (en-AU)": f"{HF_USER}/besstie-sarcasm-gemma2b-lora-en-au","Indian (en-IN)":    f"{HF_USER}/besstie-sarcasm-gemma2b-lora-en-in",}
# Adapter names registered inside the model (used by set_adapter)
ADAPTER_NAMES = {"British (en-UK)":   "en-UK","Australian (en-AU)": "en-AU","Indian (en-IN)": "en-IN",}
MAX_LENGTH = 384  # matches Stage 5 training
# model loading (one-time, at startup)
print("BESSTIE Sarcasm Detector starting up")
print(f"Loading tokenizer ({BASE_MODEL})")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"[2/3] Loading base model in 4-bit (Gemma-2-2B)")
bnb_config = BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16,)
t0 = time.time()
base_model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL,num_labels=2,quantization_config=bnb_config,torch_dtype=torch.float16,device_map="auto",)
base_model.config.pad_token_id = tokenizer.pad_token_id
print(f"base model loaded in {time.time()-t0:.1f}s")
print(f"[3/3] Attaching 3 LoRA adapters from HF Hub")
model = None
for label, repo_id in ADAPTER_REPOS.items():
    name = ADAPTER_NAMES[label]
    t0 = time.time()
    if model is None:
        model = PeftModel.from_pretrained(base_model, repo_id, adapter_name=name)
    else:
        model.load_adapter(repo_id, adapter_name=name)
    print(f"{label:<22} {repo_id}({time.time()-t0:.1f}s)")
model.eval()
print(f"Ready. VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")
print(f"Registered adapters: {list(model.peft_config.keys())}")

# -------- prediction function --------
def predict(text, variety_label):
    if not text or not text.strip():
        return "—", 0.0, "Please enter some text."
    if variety_label not in ADAPTER_NAMES:
        return "—", 0.0, f"Unknown variety: {variety_label}"
    adapter_name = ADAPTER_NAMES[variety_label]
    model.set_adapter(adapter_name)
    t0 = time.time()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH,).to(model.device)
    with torch.no_grad():
        out = model(**inputs)
    probs = torch.softmax(out.logits.float(), dim=-1).cpu().numpy()[0]
    p_not_sarc, p_sarc = float(probs[0]), float(probs[1])
    pred = int(p_sarc >= 0.5)
    latency_ms = (time.time() - t0) * 1000
    label = "Sarcastic" if pred == 1 else "Not sarcastic"
    confidence = p_sarc if pred == 1 else p_not_sarc
    info = (f"Variety: **{variety_label}** (adapter: `{adapter_name}`) · "f"P(sarcastic) = {p_sarc:.3f} · "f"P(not sarcastic) = {p_not_sarc:.3f} · "f"latency: {latency_ms:.0f} ms")
    return label, confidence, info
# example texts (one per variety, drawn from BESSTIE-style content)
EXAMPLES = [
    ["Oh fantastic, another rail strike. Just what I needed this Monday morning.", "British (en-UK)"],
    ["Yeah nah, the LNP really nailed it with this budget. Brilliant stuff.", "Australian (en-AU)"],
    ["Wah, the new traffic rules are working so beautifully. Absolutely no jams now.", "Indian (en-IN)"],
    ["I genuinely enjoyed the film — the cinematography was stunning.", "British (en-UK)"],
    ["The match yesterday was a cracker, both teams played really well.", "Australian (en-AU)"],
    ["The food at the new restaurant is quite good, would recommend.", "Indian (en-IN)"],
]
# Gradio UI
with gr.Blocks(title="BESSTIE Sarcasm Detector", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        #BESSTIE Sarcasm Detector
        Detect sarcasm in **British, Australian, and Indian English** using
        variety-specific LoRA adapters on Gemma-2-2B.
        Select the English variety closest to the text's origin — the
        adapter swap happens in milliseconds because we ship tiny LoRA
        deltas (≈25 MB each) on top of one shared base model.
        """
    )
    with gr.Row():
        with gr.Column(scale=2):
            text_in = gr.Textbox(label="Text to classify",placeholder="Type or paste a sentence…",lines=4,)
            variety_in = gr.Dropdown(choices=list(ADAPTER_NAMES.keys()),value="British (en-UK)",label="English variety",)
            predict_btn = gr.Button("Detect sarcasm", variant="primary")
        with gr.Column(scale=1):
            label_out = gr.Label(label="Prediction")
            confidence_out = gr.Slider(label="Confidence", minimum=0, maximum=1, value=0,interactive=False,)
            info_out = gr.Markdown(label="Details")
    gr.Examples(examples=EXAMPLES,inputs=[text_in, variety_in],label="Try these examples",)
    gr.Markdown(
        """
        ---
        **About** — The model is Gemma-2-2B (4-bit quantized) with one of three
        LoRA adapters fine-tuned on the
        [BESSTIE](https://huggingface.co/datasets/surrey-nlp/BESSTIE-CW-26)
        sarcasm dataset. Adapters are hosted on HuggingFace Hub
        (`Mohammadeeu20/besstie-sarcasm-gemma2b-lora-*`). Built for the
        COMM061 NLP coursework at the University of Surrey, 2026.
        """
    )
    predict_btn.click(fn=predict,inputs=[text_in, variety_in],outputs=[label_out, confidence_out, info_out],)
if __name__ == "__main__":
    demo.launch(server_name="127.0.0.1",server_port=7860,share=False,show_error=True,)
'''

In [ ]:
APP_PATH.write_text(APP_CODE)
print(f"Wrote {APP_PATH} ({len(APP_CODE)} chars, {len(APP_CODE.splitlines())} lines)")
result = subprocess.run(["pip", "show", "gradio"],capture_output=True, text=True)
if result.returncode == 0:
    version_line = [l for l in result.stdout.split("\n") if l.startswith("Version:")][0]
    print(f"Gradio: {version_line.strip()}")
else:
    print("Gradio NOT installed install with: pip install gradio")

### Stage 7B Observations
-Architecture: single Gemma-2-2B base model loaded in 4-bit (~2.4 GB VRAM) with three LoRA adapters (en-UK / en-AU / en-IN) registered to the same base via load_adapter. A single set_adapter(name) call (sub-milliseconds) is called variety switching.
-Adapter hosting: three adapters were published on HuggingFace Hub at Mohammadeeu20/besstie-sarcasm-gemma2b-lora-{variety). The submission ZIP includes application code only, adapters are downloaded at runtime.
-Deployment efficiency: 2.5GB total VRAM (base + 3 adapters) vs. about 16.5GB in three independent full-precision Gemma instances. Request-based latency 50- 85 ms on GPU.
-Single-input demonstration of the variety gap: same input ("Yeah nah, the LNP really nailed it... The cross-variety transfer pattern of Stage 4 appears at single-input granularity.

## Stage 8: Efficiency

In [ ]:
# Sarcasm task, all varieties pooled (like Stage 2.1 setup)
X_train = train_df["text"].tolist()
y_train = train_df["Sarcasm"].astype(int).to_numpy()
X_test  = test_df["text"].tolist()
y_test  = test_df["Sarcasm"].astype(int).to_numpy()

In [ ]:
print(f"Training:{len(X_train)} examples")
print(f"Test:{len(X_test)} examples")

In [ ]:
t0 = time.perf_counter()
tfidf_vec = TfidfVectorizer(ngram_range=(1, 2),min_df=2,sublinear_tf=True,)

In [ ]:
X_train_tfidf = tfidf_vec.fit_transform(X_train)
X_test_tfidf  = tfidf_vec.transform(X_test)
fit_time_vec = time.perf_counter() - t0
print(f"TF-IDF vectorizer fit:{fit_time_vec*1000:.1f} ms")
print(f"vocabulary size:{len(tfidf_vec.vocabulary_):,} terms")

In [ ]:
t0 = time.perf_counter()
lr_model = LogisticRegression(class_weight="balanced",max_iter=2000,random_state=42,n_jobs=-1,)

In [ ]:
lr_model.fit(X_train_tfidf, y_train)
fit_time_lr = time.perf_counter() - t0
print(f"LogReg fit:{fit_time_lr*1000:.1f} ms")

In [ ]:
# checking macro-F1 matches Stage 2 result
y_pred = lr_model.predict(X_test_tfidf)
m_f1 = f1_score(y_test, y_pred, average="macro")
c1_f1 = f1_score(y_test, y_pred, pos_label=1)
print(f"Macro-F1: {m_f1:.4f}   Class-1 F1: {c1_f1:.4f}")
print(f"Stage 2 logged: Macro-F1 = 0.621   Class-1 F1 = 0.383")

In [ ]:
# for stage 8m retrain RoBERTa-base sarcasm pooled
ROBERTA_NAME = "roberta-base"
SCRATCH_RUNS = PROJECT_ROOT / "_runs"
SCRATCH_RUNS.mkdir(exist_ok=True)

In [ ]:
# matches the stage 3 implementation
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        if class_weights is not None:
            self.class_weights = class_weights.to(self.args.device)
        else:
            self.class_weights = None
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels").long()  # cast to long for CE
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
def compute_metrics_hf(eval_pred):
    preds = eval_pred.predictions.argmax(axis=-1)
    labels = eval_pred.label_ids
    p, r, f, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {"macro_f1": f, "macro_precision": p, "macro_recall": r}

In [ ]:
roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_NAME)

In [ ]:
def tok_roberta(batch):
    return roberta_tokenizer(batch["text"], truncation=True, max_length=384, padding=False)

In [ ]:
train_ds_r = Dataset.from_pandas(train_df[["text", "Sarcasm"]].rename(columns={"Sarcasm": "labels"}),preserve_index=False,).map(tok_roberta, batched=True, remove_columns=["text"])

In [ ]:
val_ds_r = Dataset.from_pandas(val_df[["text", "Sarcasm"]].rename(columns={"Sarcasm": "labels"}),preserve_index=False,).map(tok_roberta, batched=True, remove_columns=["text"])

In [ ]:
# class weights (balanced)
y_tr = np.array(train_ds_r["labels"])
weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_tr)
class_weights_t = torch.tensor(weights, dtype=torch.float)
print(f"class weights: {weights}")

In [ ]:
roberta_model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_NAME, num_labels=2)

In [ ]:
run_dir = SCRATCH_RUNS / "stage8_roberta_sarcasm"
if run_dir.exists():
    shutil.rmtree(run_dir)

In [ ]:
args = TrainingArguments(output_dir=str(run_dir),num_train_epochs=4,learning_rate=2e-5,per_device_train_batch_size=16,per_device_eval_batch_size=32,warmup_ratio=0.1,weight_decay=0.01,logging_steps=50,eval_strategy="epoch",save_strategy="epoch",save_total_limit=1,load_best_model_at_end=True,metric_for_best_model="macro_f1",greater_is_better=True,fp16=True,seed=42,data_seed=42,report_to="none",disable_tqdm=False,)

In [ ]:
trainer = WeightedTrainer(class_weights=class_weights_t,model=roberta_model,args=args,train_dataset=train_ds_r,eval_dataset=val_ds_r,processing_class=roberta_tokenizer,data_collator=DataCollatorWithPadding(roberta_tokenizer),compute_metrics=compute_metrics_hf,)

In [ ]:
t0 = time.time()
trainer.train()
train_time = time.time() - t0
print(f"raining done in {train_time:.1f}s ({train_time/60:.1f} min)")

In [ ]:
# best model is loaded into trainer.model
roberta_model = trainer.model
roberta_model.eval()

In [ ]:
test_ds_r = Dataset.from_pandas(test_df[["text", "Sarcasm"]].rename(columns={"Sarcasm": "labels"}),preserve_index=False,).map(tok_roberta, batched=True, remove_columns=["text"])

In [ ]:
preds_out = trainer.predict(test_ds_r)
y_pred = preds_out.predictions.argmax(axis=-1)
y_true = preds_out.label_ids
m_f1 = f1_score(y_true, y_pred, average="macro")
c1_f1 = f1_score(y_true, y_pred, pos_label=1)
print(f"Test Macro-F1: {m_f1:.4f}   Class-1 F1: {c1_f1:.4f}")

In [ ]:
# cleanup checkpoint dir but keep the model in memory
if run_dir.exists():
    shutil.rmtree(run_dir)
print(f"Cleaned up checkpoint dir: {run_dir}")
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

In [ ]:
LLM_NAME = "google/gemma-2-2b"
ADAPTER_DIR = PROJECT_ROOT / "_adapters"
# 4-bit config, matches Stage 5
bnb_config = BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16,)

In [ ]:
# Gemma tokenizer
print("Loading Gemma tokenizer...")
gemma_tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
if gemma_tokenizer.pad_token is None:
    gemma_tokenizer.pad_token = gemma_tokenizer.eos_token

In [ ]:
# Gemma base 4-bit
t0 = time.time()
gemma_base = AutoModelForSequenceClassification.from_pretrained(LLM_NAME,num_labels=2,quantization_config=bnb_config,torch_dtype=torch.float16,device_map="auto",)
gemma_base.config.pad_token_id = gemma_tokenizer.pad_token_id
print(f"  loaded in {time.time()-t0:.1f}s")

In [ ]:
# attach en-AU seed 123 adapter
adapter_path = ADAPTER_DIR / "en-AU_seed123"
print(f"Attaching LoRA adapter from {adapter_path}")
gemma_lora = PeftModel.from_pretrained(gemma_base, str(adapter_path))
gemma_lora.eval()

In [ ]:
# 50 examples with a fixed seed so timings are reproducible
rng = np.random.default_rng(42)
sample_indices = rng.choice(len(test_df), size=50, replace=False)
sample_texts = test_df.iloc[sample_indices]["text"].tolist()

In [ ]:
print(f"Sample of 50 texts word count:"
      f"min={min(len(t.split()) for t in sample_texts)}, "
      f"max={max(len(t.split()) for t in sample_texts)}, "
      f"mean={np.mean([len(t.split()) for t in sample_texts]):.1f}")

In [ ]:
N_TRIALS = 5  # 1 warmup + 4 measured

In [ ]:
# run fn() n_trials times and discard first (warmup), return measured times
def time_function(fn, n_trials=N_TRIALS, sync_cuda=False):
    times = []
    for trial in range(n_trials):
        if sync_cuda:
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn()
        if sync_cuda:
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0
        times.append(elapsed)
    return times[1:]  # discard warmup

In [ ]:
# single input
def tfidf_single():
    for text in sample_texts:
        x = tfidf_vec.transform([text])
        _ = lr_model.predict(x)

In [ ]:
# TF-IDF is fast enough that batch=32 is sort of meaningless, so im gonna use batch of 50
def tfidf_batch():
    x = tfidf_vec.transform(sample_texts)
    _ = lr_model.predict(x)

In [ ]:
t_single = time_function(tfidf_single, sync_cuda=False)
t_batch  = time_function(tfidf_batch,  sync_cuda=False)
# Per example averages
single_per_ex_ms = [t / len(sample_texts) * 1000 for t in t_single]
batch_per_ex_ms  = [t / len(sample_texts) * 1000 for t in t_batch]

In [ ]:
print(f"Single input (1 text at a time, 50 examples):")
print(f"Per-example latency: median = {np.median(single_per_ex_ms):.3f} ms  "
      f"(range {min(single_per_ex_ms):.3f}--{max(single_per_ex_ms):.3f})")
print(f"Batch input (50 texts at once):")
print(f"Per-example latency: median = {np.median(batch_per_ex_ms):.4f} ms  "
      f"(range {min(batch_per_ex_ms):.4f}--{max(batch_per_ex_ms):.4f})")

In [ ]:
tfidf_results = {"model": "TF-IDF + LogReg","single_per_ex_ms": float(np.median(single_per_ex_ms)),"batch_per_ex_ms":  float(np.median(batch_per_ex_ms)),"speedup":float(np.median(single_per_ex_ms) / np.median(batch_per_ex_ms)),}

In [ ]:
device = next(roberta_model.parameters()).device
print(f"device: {device}")

In [ ]:
def roberta_single():
    for text in sample_texts:
        inputs = roberta_tokenizer(text, return_tensors="pt", truncation=True, max_length=384).to(device)
        with torch.no_grad():
            _ = roberta_model(**inputs)

In [ ]:
def roberta_batch():
    inputs = roberta_tokenizer(
        sample_texts, return_tensors="pt", truncation=True, max_length=384, padding=True,
    ).to(device)
    with torch.no_grad():
        _ = roberta_model(**inputs)

In [ ]:
t_single = time_function(roberta_single, sync_cuda=True)
t_batch  = time_function(roberta_batch,  sync_cuda=True)
single_per_ex_ms = [t / len(sample_texts) * 1000 for t in t_single]
batch_per_ex_ms  = [t / len(sample_texts) * 1000 for t in t_batch]

In [ ]:
print(f"Single input (1 text at a time, 50 examples):")
print(f"Per-example latency: median = {np.median(single_per_ex_ms):.2f} ms  "
      f"(range {min(single_per_ex_ms):.2f}--{max(single_per_ex_ms):.2f})")
print(f"Batch input (50 texts at once):")
print(f"Per-example latency: median = {np.median(batch_per_ex_ms):.2f} ms  "
      f"(range {min(batch_per_ex_ms):.2f}--{max(batch_per_ex_ms):.2f})")

In [ ]:
roberta_results = {"model": "RoBERTa-base","single_per_ex_ms": float(np.median(single_per_ex_ms)),"batch_per_ex_ms":  float(np.median(batch_per_ex_ms)),"speedup":float(np.median(single_per_ex_ms) / np.median(batch_per_ex_ms)),}

In [ ]:
device = next(gemma_lora.parameters()).device
print(f"device: {device}")

In [ ]:
def gemma_single():
    for text in sample_texts:
        inputs = gemma_tokenizer(text, return_tensors="pt", truncation=True, max_length=384).to(device)
        with torch.no_grad():
            _ = gemma_lora(**inputs)

In [ ]:
def gemma_batch():
    inputs = gemma_tokenizer(
        sample_texts, return_tensors="pt", truncation=True, max_length=384, padding=True,
    ).to(device)
    with torch.no_grad():
        _ = gemma_lora(**inputs)

In [ ]:
t_single = time_function(gemma_single, sync_cuda=True)
t_batch  = time_function(gemma_batch,  sync_cuda=True)
single_per_ex_ms = [t / len(sample_texts) * 1000 for t in t_single]
batch_per_ex_ms  = [t / len(sample_texts) * 1000 for t in t_batch]

In [ ]:
print(f"Single input (1 text at a time, 50 examples):")
print(f"Per-example latency: median = {np.median(single_per_ex_ms):.2f} ms  "
      f"(range {min(single_per_ex_ms):.2f}--{max(single_per_ex_ms):.2f})")
print(f"Batch input (50 texts at once):")
print(f"Per-example latency: median = {np.median(batch_per_ex_ms):.2f} ms  "
      f"(range {min(batch_per_ex_ms):.2f}--{max(batch_per_ex_ms):.2f})")

In [ ]:
print(f"Single input (1 text at a time, 50 examples):")
print(f"Per-example latency: median = {np.median(single_per_ex_ms):.2f} ms  "
      f"(range {min(single_per_ex_ms):.2f}--{max(single_per_ex_ms):.2f})")
print(f"Batch input (50 texts at once):")
print(f"Per-example latency: median = {np.median(batch_per_ex_ms):.2f} ms  "
      f"(range {min(batch_per_ex_ms):.2f}--{max(batch_per_ex_ms):.2f})")

In [ ]:
gemma_results = {"model": "Gemma-2-2B + LoRA (4-bit)","single_per_ex_ms": float(np.median(single_per_ex_ms)),"batch_per_ex_ms": float(np.median(batch_per_ex_ms)),"speedup":float(np.median(single_per_ex_ms) / np.median(batch_per_ex_ms)),}


In [ ]:
all_timings = [tfidf_results, roberta_results, gemma_results]
print(f"{'Model':<28}  {'Single (ms/ex)':>15}  {'Batch (ms/ex)':>15}  {'Batch speedup':>15}")
for r in all_timings:
    print(f"{r['model']:<28}  {r['single_per_ex_ms']:>15.3f}  {r['batch_per_ex_ms']:>15.3f}  {r['speedup']:>14.1f}x")

In [ ]:
out_path = PROJECT_ROOT / "results/q5_2_efficiency.json"
with open(out_path, "w") as f:
    json.dump({"n_examples": len(sample_texts),"n_trials":N_TRIALS,"n_measured": N_TRIALS - 1,"metric":"median over measured trials","results":all_timings,},f, indent=2)
print(f"Saved {out_path}")

In [ ]:
# get macro-F1 from logged results (seed 42 for parity with the timing run)
def get_macro_f1(model_str, seed_val=None):
    sub = df[(df["model"] == model_str) & (df["task"] == "Sarcasm")]
    if seed_val is not None:
        sub = sub[sub["seed"] == seed_val]
    if "training_data" in sub.columns:
        sub = sub[sub["training_data"] == "all_varieties_pooled"]
    return sub["macro_f1"].mean() if len(sub) else None

In [ ]:
f1_tfidf   = get_macro_f1("tfidf_logreg")
f1_roberta = get_macro_f1("roberta_base")
# for Gemma use the en-AU diagonal since thats what was timed
f1_gemma   = df[(df["model"] == "gemma2b_lora")& (df["train_variety"] == "en-AU")& (df["test_variety"]  == "en-AU")& (df["seed"] == 123)]["macro_f1"].iloc[0]

In [ ]:
print(f"TF-IDF + LR:{f1_tfidf:.4f}")
print(f"RoBERTa-base (pooled):{f1_roberta:.4f}")
print(f"Gemma + LoRA (en-AU s123):{f1_gemma:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
models  = ["TF-IDF + LR", "RoBERTa-base", "Gemma + LoRA (4-bit)"]
single_ms = [tfidf_results["single_per_ex_ms"],roberta_results["single_per_ex_ms"],gemma_results["single_per_ex_ms"]]
batch_ms  = [tfidf_results["batch_per_ex_ms"],roberta_results["batch_per_ex_ms"],gemma_results["batch_per_ex_ms"]]
f1_vals   = [f1_tfidf, f1_roberta, f1_gemma]
colors    = ["#2ca02c", "#1f77b4", "#d62728"]
# filled circles = single input latency
for x, y, c, m in zip(single_ms, f1_vals, colors, models):
    ax.scatter(x, y, s=200, c=c, marker="o", edgecolors="black",linewidths=1.5, zorder=3, label=f"{m} (single)")
# open squares = batch latency (same colour)
for x, y, c, m in zip(batch_ms, f1_vals, colors, models):
    ax.scatter(x, y, s=200, c="white", marker="s", edgecolors=c,linewidths=2, zorder=3, label=f"{m} (batch)")
# arrows show the batching benefit per model
for s2, b, y, c in zip(single_ms, batch_ms, f1_vals, colors):
    ax.annotate("", xy=(b, y), xytext=(s2, y),arrowprops=dict(arrowstyle="->", color=c, lw=1.5, alpha=0.5))
for x, y, m, c in zip(single_ms, f1_vals, models, colors):
    ax.annotate(m, (x, y), xytext=(8, 8), textcoords="offset points",fontsize=10, color=c, fontweight="bold")
ax.set_xscale("log")
ax.set_xlabel("Per-example inference latency (ms, log scale)", fontsize=11)
ax.set_ylabel("Sarcasm Macro-F1", fontsize=11)
ax.set_title(
    "Figure 9 Accuracy vs latency Pareto frontier\n"
    "Filled circles: single-input latency. Open squares: batch (50) latency. "
    "Arrows show batching benefit per model.",
    fontsize=11,
)
ax.grid(True, alpha=0.3, which="both")
ax.set_ylim(0.55, 0.85)

### Stage 8 — Observations
-The latency difference between architectures is two orders of magnitude ((0.18 ms TF-IDF → 39.2 ms Gemma+LoRA single) though the Macro-F1 range is only 0.17 (0.621 → 0.793). Accuracy efficiency trade-off is heavily decreasing.
-Batching is beneficial to disproportionately benefit simple models: TF-IDF improves 6.8 times through batching, RoBERTa 3.8 times, Gemma+LoRA only 2.0 times. Larger models congest the GPU with single-batch, with minimal amortisation.
-Operating-point recommendations:
  Single-request interactive use, accuracy-critical → Gemma + LoRA (~40 ms, best Macro-F1)
  Scale to high-throughput batch processing → RoBERTa-base (~0.9 ms/example in batch, latency in transformer accuracy near-classical)
  CPU-only / embedded → TF-IDF + LR (sub-milliseconds, accept Macro-F1 penalty)
-Methodology:time.perf_counter() with torch.cuda.synchronize() before/after GPU operations to avoid measuring the submissions to the async queue. Mean of 4 trials measured, 1 warmup discarded.